# 04 · Complete issue and evidence logs

**PRT2030 · Dataset inspection · Study notebook**

This notebook turns the checks from Notebooks 01–03 into one complete inspection record.
It loads `wildfires.csv` independently, creates a Pandas DataFrame for every log and evidence
table, and writes the tables to separate sheets in `Wildfires_Inspection_Logs.xlsx`.
Run from the top. The dataset should be beside the notebook, as in the first three notebooks.

## Libraries to install before running

| Package | Required here? | Purpose |
| --- | --- | --- |
| [pandas](https://pandas.pydata.org/docs/) | Yes; use **3.0.5** for this module | Load the CSV, perform the checks and build the log DataFrames. |
| [NumPy](https://numpy.org/doc/stable/) | Yes; normally installed with Pandas | Identify finite and infinite numerical values. |
| [openpyxl](https://openpyxl.readthedocs.io/en/stable/) | Yes; the main additional package | Write the Excel workbook, apply formatting and check the saved sheets. |
| [ipykernel](https://ipykernel.readthedocs.io/en/stable/) | Needed for running notebooks in VS Code; you may already have it | Provide the Python notebook kernel. It also supplies IPython for displaying tables. |
| [lxml](https://lxml.de/) | Optional; recommended for the large wildfire file | Speed up the memory-saving Excel writer. The notebook works without it. |

In the VS Code terminal, using the Python interpreter selected for your notebook:

```bash
python -m pip install "pandas==3.0.5" numpy openpyxl ipykernel
```

Optional installation for the full dataset:

```bash
python -m pip install lxml
```

If the terminal uses a different Python, run this command in a **temporary notebook code cell**
so that installation targets the active notebook kernel:

```python
%pip install "pandas==3.0.5" numpy openpyxl ipykernel
```

Restart the kernel after installation. In VS Code, the Python and Jupyter extensions must also
be enabled. Matplotlib is not needed by this fourth notebook. `csv`, `pathlib`, `hashlib`,
`datetime`, `math`, `re` and `sys` are part of Python and require no separate installation.

**Reading the examples:** The small values introduced as examples explain the logic;
they are not new measurements from the wildfire file. Comments explain the purpose,
intermediate results and reasons for the choices. Apply the stated prerequisites
before interpreting a zero flag count as a pass.


## 1. Understand the records we will create

| Table | What it records | Meaning of one row |
| --- | --- | --- |
| `issue_log` | A finding, its analytical effect and a proposed next step | One issue, even if many source records support it |
| `check_log` | The rule, eligible population, measured result and outcome of every check | One inspection check |
| An evidence DataFrame | Original values and any comparison values needed to reproduce a finding | Usually one flagged source record |
| A summary DataFrame | Counts or statistical context | One column, category, year, combination or statistic, as labelled |

The process is: **state a question → check the data → keep evidence → record the result → explain a decision**.
A passed check belongs in the check log. It does not need an invented issue.
An empty evidence table keeps its headings and is exported so that a zero result remains visible.

We retain M01, M02, C01, R01, R02, T01, T02, T03, V01, V02, L01 and O01 from the study notebooks.
S01 and P01–P04 remain check references. Suffixes distinguish checks within a family, such as
`P01_size_parse`. New references label previously unnumbered checks: S02–S04 for headers and
column variation, M03 for entirely missing rows, T04 for distinct spelling variants, K01–K04
for categories/codes and N01–N03 for numerical meaning. Missing fields beyond M01/M02 receive
readable references such as `M_state`.

A check can be **Passed**, **Flagged for review**, **Untested** or **Not applicable**.
Passed applies only to its eligible population. A zero eligible count produces Untested.
All checks here apply to the wildfire schema; Not applicable is available when adapting the helper.
A flagged value, a coverage gap, a repeated row or an outlier is an observation to investigate.
It is not automatically a confirmed error or a reason to delete a record.

**A check-log rule in concrete terms:** “Among finite converted years, flag a fractional year”
tests `1998.5` and flags it, tests `1998` and accepts it, but cannot test a missing year for
numeric meaning. The check log records the checked population and count even if it finds no
fractional years; an issue entry is needed only when a flagged finding has analytical judgement.


### 1.1 Load the original dataset and identify this run

**Inspection question:** Which exact file and population are these logs describing?
The file fingerprint identifies the source bytes. `source_row` is the zero-based Pandas row label
from this unchanged import, not a fire identifier or a physical CSV line number.
All counts in this notebook are computed from the file you run. Reference figures are not inserted as results.


**Documentation:** [pd.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [pd.util.hash_pandas_object](https://pandas.pydata.org/docs/reference/api/pandas.util.hash_pandas_object.html).


In [ ]:
# An inspection record should identify the exact source and preserve a starting point against
# which we can later check that inspection left the source unchanged.

# Pandas handles the tables, NumPy the finite-number checks and openpyxl the workbook. The
# remaining imports support source identification, paths and export helpers; importing them
# performs no inspection yet.
import pandas as pd
import numpy as np
import openpyxl
import csv
import hashlib
import math
import re
import sys
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

# Keep the filenames together so the same source and output are used throughout. A Path
# object supports opening a file, changing a filename and replacing the completed output.
DATASET_PATH = Path("wildfires.csv")
OUTPUT_PATH = Path("Wildfires_Inspection_Logs.xlsx")

# The SHA-256 fingerprint depends on the source bytes. Recording it lets us distinguish an
# inspection of this exact file from one of a similarly named file with changed contents.
source_hasher = hashlib.sha256()
with DATASET_PATH.open("rb") as source:
    # Read one 1-MiB block per call rather than loading another full copy of the CSV. The
    # lambda is the no-argument reading operation; iter repeats it until the empty byte
    # string b"" signals end-of-file. Each update adds the next block to the same running
    # fingerprint.
    for block in iter(lambda: source.read(1024 * 1024), b""):
        source_hasher.update(block)
# Convert the finished fingerprint to readable text and record the run time in UTC. Together
# these identify which bytes were inspected and when the record was created.
source_sha256 = source_hasher.hexdigest()
run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")

# Keep this table as the original import. Record both its dimensions and row-content hashes,
# including index labels, so later verification can detect value or label changes as well as
# added or removed rows.
raw = pd.read_csv(DATASET_PATH)
total_records = len(raw)
original_shape = raw.shape
original_content_hash = pd.util.hash_pandas_object(raw, index=True)

# Use the documented fields as prerequisites. Expected minus observed identifies required
# omissions; the reverse identifies additions that may need their own checks.
expected_columns = [
    "fire_name", "state", "discovery_date", "contained_date",
    "fire_year", "fire_size", "fire_class", "fire_cause"
]
missing_columns = sorted(set(expected_columns) - set(raw.columns))
extra_columns = sorted(set(raw.columns) - set(expected_columns))
print("Source records:", total_records)
print("Pandas:", pd.__version__, "| NumPy:", np.__version__, "| openpyxl:", openpyxl.__version__)

# Stop before dependent checks if a required field is missing. A later KeyError would be less
# helpful because it would not explain the structural problem.
if missing_columns:
    raise ValueError(f"Required fields are missing: {missing_columns}. Resolve the import first.")

# The evidence builder creates issue_id and source_row. If those labels already belonged to
# source fields, inserting references could overwrite meaning, so detect the collision before
# creating evidence.
if set(raw.columns) & {"issue_id", "source_row"}:
    raise ValueError("Source fields use reserved evidence labels: issue_id or source_row.")

# Show all selected columns but keep long text previews manageable. These settings change
# notebook display, not the evidence retained or exported.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)


### 1.2 State the rules before looking at the answers

**Inspection question:** What independent expectations will the checks use?
The same field definitions, ranges and boundaries are used in Notebooks 01–03.
Edit these rules only when the dataset documentation or the analysis question justifies the change.
The placeholder list and 25% annual-change threshold are screening choices, not proof of errors.

**Rules and screens used below — illustrative examples:**

| Expectation or screen | Accepted / unflagged example | Flagged example | Meaning of the decision |
| --- | --- | --- | --- |
| Required fields | All eight documented headings arrive | `contained_date` heading absent | A prerequisite for later checks is missing. |
| Allowed jurisdiction | `CA` | `ZZ` | Membership in this source edition's code list. |
| Two-uppercase-letter shape | `CA` or `ZZ` | `ca` or `CAA` | Written shape only; `ZZ` still fails membership. |
| Whole reported year | Finite `1998` | Finite `1998.5` | A calendar-year value should not be fractional. |
| Extract period | `1992` through `2015` | `2020` | Applies to this extract, not every wildfire file. |
| Positive finite size | `0.1` acres | Finite `0` or `-2` acres | The measurement's meaning before class/distribution checks. |
| Expected size class | 10 acres reported as `C` | 10 acres reported as `B` | Two fields can disagree despite being individually usable. |
| Date order | Discovery 10 July, containment 12 July | Containment 9 July | Negative duration would result from the flagged pair. |
| Surrounding whitespace | `"Oak"` unchanged | `" Oak "` becomes `"Oak"` | A text-format difference, not confirmed incident duplication. |
| Placeholder candidates | `"Oak"` | `" UNKNOWN "` matches candidate `unknown` | Investigate meaning; this is a screening choice. |
| Annual-change review | 100 → 124 records: +24% | 100 → 130: +30%; 100 → 70: -30% | At least 25% in either direction requests context. |
| IQR review | 11 acres at an illustrative upper fence of 11 | 12 acres above that fence | Statistical review, not proof of an error. |

**Prerequisites:** A missing name cannot enter a spelling comparison; a non-finite year cannot
enter a finite-year meaning check; a missing or unreadable endpoint cannot enter date order.
These cases are **untested for those rules** and remain visible in the earlier checks.
A missing-field check itself can inspect every source record, including those with missing values.


In [ ]:
# Declare independent expectations and classroom screening choices before seeing which values
# occur, then prepare containers for the results.

# Membership examples: CA is allowed; ZZ is not. A two-letter shape is insufficient for
# membership. Use the source reference list rather than observed codes, or every observed
# code would automatically pass.
allowed_states = (
    "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA "
    "MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD "
    "TN TX UT VT VA WA WV WI WY DC PR"
).split()
allowed_classes = ["A", "B", "C", "D", "E", "F", "G"]
# Missing/Undefined is an allowed source label, but it does not identify a known cause.
# Category validity and information usefulness therefore need different checks.
allowed_causes = [
    "Lightning", "Equipment Use", "Smoking", "Campfire", "Debris Burning",
    "Railroad", "Arson", "Children", "Miscellaneous", "Fireworks",
    "Powerline", "Structure", "Missing/Undefined"
]
# 1992-2015 is this extract's period: range excludes the stop year 2016. A whole 2020 year
# would still lie outside this period.
expected_years = list(range(1992, 2016))
# These words are screening candidates, not automatically invalid names. Trimming and
# casefolding will let " UNKNOWN " match unknown, after which its meaning still needs review.
placeholder_words = ["unknown", "unnamed", "not named", "none"]
# The conversion format checks whether text can supply a calendar value. The separate pattern
# checks digit/separator layout: month 99 can match the layout while failing calendar
# conversion.
DATE_FORMAT = "%Y-%m-%d %H:%M:%S"
TIMESTAMP_PATTERN = r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"
# A rise from 100 to 130 or fall to 70 records meets the 25% classroom review threshold. The
# 1.5-IQR multiplier sets a distribution screen; neither choice proves a source value is
# wrong.
ANNUAL_CHANGE_THRESHOLD = 25
IQR_MULTIPLIER = 1.5

# Accumulate dictionaries here because each completed dictionary will become ONE log row.
# Starting with empty lists on a full restart prevents results from an earlier run mixing
# with the new run.
issue_entries = []
check_entries = []

# Map a table name to its actual DataFrame so export can find every result. evidence_rules
# separately records how to reconcile that table with its flagged count: all rows, or True
# values in a named flag column.
tables = {}
evidence_rules = {}

# Fixed headings keep log structure consistent even when no issues are found. Different
# checks can then be assembled and exported without changing the interpretation of a column.
ISSUE_COLUMNS = [
    "issue_id", "finding_type", "finding", "field", "affected_count",
    "checked_count", "count_unit", "percent", "question", "expectation",
    "check_reference", "evidence_table", "evidence_scope", "impact",
    "severity", "severity_reason", "proposed_action", "status", "verification"
]
CHECK_COLUMNS = [
    "check_id", "check", "field", "total_count", "checked_count", "not_checked_count",
    "flagged_count", "count_unit", "percent", "outcome", "check_reference",
    "evidence_table", "evidence_scope", "notes"
]


### 1.3 Use two small helpers to keep the repeated steps consistent

The inspection rules remain visible in each investigation below. These helpers only organise
evidence and log entries. Read their comments once, then concentrate on each check's question and rule.
A function definition creates the helper; it does not run a check until we call it.

`make_evidence` selects **every** flagged record and adds its source reference. A ten-row display
later in the notebook is only a preview. We never replace the full table with that preview.

**What the helpers do with a rule:** Suppose the visible whitespace condition flags source rows
17 and 42. The evidence helper keeps those two original rows and their comparisons. The log
helper records two flags out of the present-name population. Neither helper decides that a name
is wrong or removes whitespace; that judgement belongs to the specific investigation.


**Documentation:** [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [DataFrame.copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [DataFrame.insert](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.insert.html).


In [ ]:
# This helper repeats evidence organisation, while each caller still supplies its own visible
# detection condition and useful source fields.

def make_evidence(flags, check_id, columns, comparisons=None):

    # This helper deliberately requires one decision for every source row, in source order. A
    # missing or mislabelled decision can change the evidence population; this strict check
    # catches a mask built from a different table before selection.
    if not flags.index.equals(raw.index):
        raise ValueError(f"{check_id}: evidence flags do not match the source rows.")
    # An unknown decision is not established evidence of a flag. Exclude it from this
    # selection; the caller must separately report prerequisites and untested inputs.
    flags = flags.fillna(False)
    # Select every True source row, then copy before adding annotations. No head or sample
    # occurs here, so the full supporting population is retained.
    evidence = raw.loc[flags, columns].copy()
    # Save the original index as ordinary data for exports that omit the index. Inserting the
    # issue label ahead of it gives each row both a finding reference and a route back to its
    # source.
    evidence.insert(0, "source_row", evidence.index)
    evidence.insert(0, "issue_id", check_id)

    # Some checks need derived values beside originals, such as a trimmed name or a discovery
    # year. None means no extra comparison was requested; otherwise each dictionary key
    # becomes a new evidence heading.
    if comparisons is not None:
        for label, values in comparisons.items():
            # Retrieve comparison values by the selected source labels. If evidence holds
            # rows 17 and 42, fetch comparison values 17 and 42; taking the first two values
            # would compare different records.
            evidence[label] = values.loc[evidence.index]
    # Return the complete new table so the caller can register it under a table name.
    # Defining the function above did not run a check; a later call supplies actual flags.
    return evidence


`record_check` always adds a check-log entry. It adds an issue-log entry only when a finding is
flagged and a judgement dictionary is supplied. The dictionary supplies the analytical question,
impact, severity reasoning, proposed action and verification; the helper supplies the measured facts.
Use **Not assigned** for severity when the evidence does not yet justify a priority.
The source is still unchanged: a proposed action is text, not executed cleaning code.

Counts use `count_unit`: usually records, but sometimes columns, distinct spellings, years or
jurisdiction–year combinations. `percent` is stored on a **0–100 scale**, matching the existing notebooks.
It is unavailable when no items were eligible. Overlapping issue counts must not be added to count unique problem records.

**Count-reconciliation example:** If evidence contains two flagged records, reconcile by its
two rows. If a complete five-column summary has only two flagged columns, reconcile by those
two True flags, not all five summary rows. R02 likewise keeps full matching groups while counting
only their newly flagged members. The counting instruction must match the meaning of the finding.


In [ ]:
# Keep check recording consistent while leaving measured facts separate from judgement. A
# passed or untested check must stay visible without inventing an issue.

def record_check(check_id, title, field, checked_count, flagged_count, evidence_table,
                 *, unit="records", total_count=None, reference="", judgement=None,
                 evidence_scope="All flagged records", notes="", reconcile="rows",
                 applicable=True):

    # checked is the eligible population and flagged is its matching subset. The * in the
    # function signature makes settings after evidence_table named arguments, helping
    # distinguish unit, reference and judgement at the call site.
    checked = int(checked_count)
    flagged = int(flagged_count)
    # For record checks, the whole-file total exposes excluded records as total minus
    # checked. For another unit, such as years, the caller can supply that unit's total;
    # adding record totals to year totals would be meaningless.
    if total_count is None:
        total_count = total_records if unit == "records" else checked
    total = int(total_count)
    # Seven flagged items out of five checked is impossible; so is checking more than the
    # total. Reject inconsistent counts instead of putting a believable-looking percentage
    # into the log.
    if not 0 <= flagged <= checked <= total:
        raise ValueError(f"{check_id}: counts must satisfy flagged <= checked <= total.")
    # Inspect existing check dictionaries for the same reference. any is True if at least one
    # matches. Rejecting a duplicate stops a partial rerun from appending the same check
    # twice while replacing its evidence table.
    if any(entry["check_id"] == check_id for entry in check_entries):
        raise ValueError(f"{check_id}: duplicate check reference. Restart and run from the top.")

    # Only divide when something was checked. Two flags out of ten checked means 20%; zero
    # out of zero has no percentage and cannot demonstrate a pass.
    percent = round(flagged / checked * 100, 4) if checked else None
    # Choose the outcome in order: irrelevant check, no eligible inputs, some flags, or
    # tested inputs with no flags. This prevents zero eligible inputs from falling into the
    # Passed branch.
    if not applicable:
        outcome = "Not applicable"
    elif checked == 0:
        outcome = "Untested"
    elif flagged > 0:
        outcome = "Flagged for review"
    else:
        outcome = "Passed"

    # Every call makes a check dictionary, including its count unit and untested population.
    # append adds it to the running check list; it does not yet create a DataFrame or write a
    # file.
    check = {
        "check_id": check_id, "check": title, "field": field,
        "total_count": total, "checked_count": checked, "not_checked_count": total - checked,
        "flagged_count": flagged, "count_unit": unit, "percent": percent,
        "outcome": outcome, "check_reference": reference,
        "evidence_table": evidence_table, "evidence_scope": evidence_scope, "notes": notes
    }
    check_entries.append(check)

    # For ordinary affected-record evidence, counting rows reconciles the result. For a full
    # five-row summary with only two True flags, validation must count that flag column
    # rather than all five rows.
    evidence_rules[check_id] = reconcile

    # An issue needs an observed flag, an applicable check and supplied analytical judgement.
    # A supporting coverage summary may have flags but no separate judgement because the main
    # coverage issue already records the concern.
    if flagged > 0 and judgement is not None and applicable:
        entry = {
            "issue_id": check_id,
            "finding_type": judgement["finding_type"],
            "finding": f"{title}: {flagged:,} of {checked:,} {unit} flagged.",
            "field": field, "affected_count": flagged, "checked_count": checked,
            "count_unit": unit, "percent": percent,
            "check_reference": reference, "evidence_table": evidence_table,
            "evidence_scope": evidence_scope, "status": "Open - decision required"
        }

        # Copy the deliberately written answers into the measured issue facts. The helper
        # cannot infer impact or priority from a percentage; those depend on the analysis
        # question supplied by the caller.
        for key in ["question", "expectation", "impact", "severity", "severity_reason",
                    "proposed_action", "verification"]:
            entry[key] = judgement[key]
        # One completed dictionary becomes one future issue row, even if its evidence table
        # contains thousands of supporting records.
        issue_entries.append(entry)


## 2. Work through one complete entry: missing containment dates — M02

**Question:** Can every source record support a discovery-to-containment duration?
Both dates are needed. A missing containment date prevents a duration for that record, even if
the reporting system allows the absence. We will first measure the finding, then keep its evidence,
then write the judgement and register the entry.


**Documentation:** [Series.isna](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [Series.sum](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html).


In [ ]:
# M02 asks whether every record can provide the containment endpoint needed for a duration
# calculation.

# The absence condition can be applied to every source record. One True marks one unavailable
# endpoint, even if that record also lacks a name or discovery date.
missing_containment = raw["contained_date"].isna()
# Sum the entire decision Series before building any preview. A plain integer count is then
# ready to record in the log.
missing_containment_count = int(missing_containment.sum())
print("Missing containment dates:", missing_containment_count)
print("Records checked:", total_records)


### 2.1 Keep the full evidence, then display a preview

One evidence row represents one affected source record. The table retains original values so
another person can return to the source. We build this first table directly to show exactly what
the evidence helper will repeat for the later investigations.


**Documentation:** [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [DataFrame.copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [DataFrame.insert](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.insert.html) · [DataFrame.head](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html).


In [ ]:
# Build the first evidence table directly so the later helper's purpose is clear: keep
# original context, traceability and all supporting records.

# Choose missing-containment rows and keep both endpoints plus context. A copy lets us
# annotate evidence without changing raw.
evidence_m02 = raw.loc[missing_containment, [
    "fire_name", "state", "discovery_date", "contained_date"
]].copy()
# The original labels remain traceable after index-free export. Inserting M02 at the front
# connects every evidence row to the same issue entry.
evidence_m02.insert(0, "source_row", evidence_m02.index)
evidence_m02.insert(0, "issue_id", "M02")
# Registration links the logical name Evidence_M02 to the actual DataFrame. Later log
# references and export use this name to find the table.
tables["Evidence_M02"] = evidence_m02
print("Rows retained in full evidence:", len(evidence_m02))
# head changes only the displayed preview; tables still holds the complete table. The
# evidence row count must equal the full M02 count.
display(evidence_m02.head(10))


### 2.2 Write the analytical judgement

**High** is justified here by the stated requirement to calculate a duration for **every** record.
This does not assign High automatically from a percentage. For another analysis question, revise
the judgement. The verification describes a later check; it does not claim cleaning has happened.


In [ ]:
# Combine the measured M02 result with an explanation of why it matters to the all-record
# duration question.

# These are written analytical answers, not operations on the source. High follows the
# requirement for every record to provide duration; it is not automatically selected by an
# affected percentage.
m02_judgement = {
    "finding_type": "Observed absence",
    "question": "Can every record support a discovery-to-containment duration?",
    "expectation": "This duration requires both usable dates.",
    "impact": "The flagged records cannot provide this duration from the available dates.",
    "severity": "High",
    "severity_reason": (
        "The stated all-record duration question cannot be met while containment dates "
        "are absent. Judge any restricted duration analysis separately."
    ),
    # Date recovery or a documented restricted population is still a proposed decision.
    # Retaining records for other questions avoids assuming unavailable duration makes every
    # other field useless.
    "proposed_action": (
        "Investigate recovery of dates. Otherwise define a documented duration subset "
        "and retain excluded records for other suitable questions."
    ),
    # Later verification must check availability, successful conversion and endpoint order: a
    # recovered string alone does not establish a usable duration.
    "verification": (
        "Repeat date availability, parsing and order checks. Report usable and excluded "
        "counts for the agreed duration population."
    )
}
# Pass measured counts and the registered evidence name to the helper. It always records the
# check, but creates the issue only if flags exist and the judgement is supplied.
record_check(
    "M02", "Missing containment dates", "contained_date", total_records,
    missing_containment_count, "Evidence_M02",
    reference='Notebook 04, section 2; Notebook 01, section 6: contained_date.isna()',
    judgement=m02_judgement
)

# A list of issue dictionaries becomes one row per issue. Transpose this display to read long
# answers vertically; an empty list keeps the specified headings if no issue was established.
display(pd.DataFrame(issue_entries, columns=ISSUE_COLUMNS).T)


## 3. Record structure, storage and completeness

**Question:** Does the file have the expected fields and usable column headings?
Keep the expected/observed schema comparison and read the original header separately: the CSV
reader can reveal repeated headings that Pandas may have renamed on import. Unexpected columns
are recorded for review; useful extra fields need not be discarded.


**Documentation:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [Series.duplicated](https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html) · [Series.str.strip](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html).


In [ ]:
# Keep a full structural comparison, then inspect original header positions so
# importer-renamed problems cannot remain hidden.

# Use expected fields plus extra observed fields so an additional column appears in the
# comparison rather than silently disappearing. One report row means one label, not one fire.
schema_labels = expected_columns + extra_columns
schema_report = pd.DataFrame({"field": schema_labels})
# For each label, record whether it was required and whether it arrived. A required present
# state has True/True; an extra temperature field has False/True.
schema_report["expected"] = schema_report["field"].isin(expected_columns)
schema_report["observed"] = schema_report["field"].isin(raw.columns)
# Different expected and observed decisions mark a structure difference. The import guard
# already stopped required omissions; any remaining added fields deserve review rather than
# automatic deletion.
schema_report["flagged"] = schema_report["expected"] != schema_report["observed"]
tables["Schema_Report"] = schema_report
# Explain why structure affects the suitability of checks. An extra field can be useful, so
# priority stays unassigned until its meaning is understood.
schema_judgement = {
    "finding_type": "Structure difference",
    "question": "Does the imported structure match the documented extract?",
    "expectation": "All required fields should be present; explain additional fields.",
    "impact": "An unexpected structure can change which checks and analyses are appropriate.",
    "severity": "Not assigned",
    "severity_reason": "Confirm the purpose of any additional field before assigning priority.",
    "proposed_action": "Check the extract definition and document any accepted extra field.",
    "verification": "Compare the final field list with the agreed extract definition."
}
# The evidence contains the complete comparison, including good labels. Reconcile by the
# flagged column; counting every report row would exaggerate structural differences.
record_check("S01", "Expected schema", "CSV columns", len(schema_report),
             schema_report["flagged"].sum(), "Schema_Report", unit="column labels",
             reference="Notebook 01, section 2.4: expected/observed field comparison",
             judgement=schema_judgement, evidence_scope="Complete expected/observed comparison",
             reconcile="flagged")

# Read the first CSV record before Pandas renaming can conceal repeated or blank headings.
# CSV parsing respects quoted commas, and utf-8-sig handles an optional leading encoding
# marker.
with DATASET_PATH.open(encoding="utf-8-sig", newline="") as source:
    header = next(csv.reader(source))
headings = pd.Series(header, dtype="string")
header_report = pd.DataFrame({"column_position": headings.index, "original_heading": headings})
# Repeated headings need all occurrences reviewed. Blank-after-trimming catches space-only
# labels; comparing originals with trimmed versions exposes " state ".
header_report["repeated"] = headings.duplicated(keep=False)
header_report["blank"] = headings.str.strip() == ""
header_report["surrounding_whitespace"] = headings != headings.str.strip()
# any(axis=1) combines the three heading concerns into one decision per position. A heading
# that is both repeated and padded counts once here, not twice.
header_report["flagged"] = header_report[["repeated", "blank", "surrounding_whitespace"]].any(axis=1)
tables["Header_Report"] = header_report
record_check("S02", "Original header problems", "CSV header", len(header_report),
             header_report["flagged"].sum(), "Header_Report", unit="column positions",
             reference="Notebook 01, section 2.6: repeated, blank or padded original headings",
             # Dictionary unpacking copies the common structural judgement, then the
             # following proposed_action overrides that one answer for header review. This
             # reuses shared reasoning while making the next step specific.
             judgement={**schema_judgement,
                        "proposed_action": "Review the original CSV headings before agreeing a correction."},
             evidence_scope="Every original header position", reconcile="flagged")


### 3.1 Keep column variation and storage summaries

**Question:** Which fields are entirely absent or contain only one observed value?
An entirely absent field supplies no observed information. A constant field cannot distinguish
records by that value. An empty source file supplies no observations for either judgement.
Dates stored as text are described here and tested through conversion later.


**Documentation:** [DataFrame.nunique](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html) · [DataFrame.notna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.notna.html) · [DataFrame.dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html).


In [ ]:
# Measure column availability and variation as different questions, using columns as the
# count unit.

# Missing values are excluded from distinct choices: [CA, CA, missing] has one observed
# value, whereas an entirely missing column has zero.
distinct_counts = raw.nunique()
present_counts = raw.notna().sum()
# These arrays are all derived in raw.columns order. to_numpy removes their field-name
# indexes so they populate the same new report positions rather than aligning against a
# different row index.
column_summary = pd.DataFrame({
    "field": raw.columns, "storage_type": raw.dtypes.astype(str).to_numpy(),
    "present_count": present_counts.to_numpy(), "distinct_present_values": distinct_counts.to_numpy()
})

# A non-empty source with zero present values supplies evidence of an entirely absent field.
# An empty file supplies no observations, so the additional total_records condition prevents
# a misleading column finding.
column_summary["entirely_missing"] = (column_summary["present_count"] == 0) & (total_records > 0)
# One choice means limited variation among observed values. It does not establish that every
# record has that value or that the field should be deleted.
column_summary["one_observed_value"] = column_summary["distinct_present_values"] == 1
tables["Column_Summary"] = column_summary
s03_judgement = {
    "finding_type": "Unavailable column information",
    "question": "Can this field contribute to the planned analysis?",
    "expectation": "A relevant field needs observed values for its planned use.",
    "impact": "An entirely absent column cannot support comparisons or calculations.",
    "severity": "Not assigned", "severity_reason": "Priority depends on the role of the affected field.",
    "proposed_action": "Investigate source coverage and decide which questions remain supportable.",
    "verification": "Recheck present values and the available population for each affected question."
}
# Check entire-column absence only when records exist. The full summary has one row per
# column, so reconcile S03 by entirely_missing rather than the report length.
record_check("S03", "Entirely missing columns", "all fields", len(column_summary) if total_records else 0,
             column_summary["entirely_missing"].sum(), "Column_Summary", unit="columns",
             total_count=len(column_summary), reference="Notebook 01, section 3.3: no observed values",
             judgement=s03_judgement, evidence_scope="Complete column summary", reconcile="entirely_missing")
# Only columns with observed values can be checked for one observed choice. Reuse the
# judgement but change its impact to limited variation; the flag column reconciles that
# different question.
record_check("S04", "One observed value in a column", "all fields", (present_counts > 0).sum(),
             column_summary["one_observed_value"].sum(), "Column_Summary", unit="columns",
             total_count=len(column_summary), reference="Notebook 01, section 3.3: nunique() == 1",
             judgement={**s03_judgement, "finding_type": "Limited column variation",
                        "impact": "The observed field cannot distinguish groups within this extract.",
                        "proposed_action": "Confirm whether a constant value is expected before retaining or excluding the field."},
             evidence_scope="Complete column summary", reconcile="one_observed_value")
display(column_summary)


### 3.2 Missing names and the remaining source fields

**Question:** Which records lack each piece of information required by an analysis?
M01 covers fire names; M02 was already recorded. The loop below creates separate checks and
evidence tables for every other source field. It includes unexpected extra fields if present.
Every percentage uses all source records for that field's missing-value check.


**Documentation:** [DataFrame.isna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) · [Series.isna](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [DataFrame.sum](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html).


In [ ]:
# Create a common completeness summary and one reproducible finding/evidence pair for each
# source field.

# The Boolean table keeps raw's shape. Summing down columns counts unavailable cells per
# field; with one cell per record in a field, that is also its affected-record count.
missing_flags = raw.isna()
missing_counts = missing_flags.sum(axis=0)
missing_summary = pd.DataFrame({
    "field": raw.columns, "missing_count": missing_counts.to_numpy(),
    "present_count": raw.notna().sum().to_numpy()
})
# Every source record can be inspected for field absence. Divide by that whole-file count,
# and keep the percentage unavailable if the source contains no records.
missing_summary["checked_count"] = total_records
missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / total_records * 100 if total_records else np.nan
)
tables["Missing_Summary"] = missing_summary

# The loop varies the source field while keeping the completeness logic consistent. Skip
# contained_date because M02 has already been measured, evidenced and registered.
for field in raw.columns:
    if field == "contained_date":
        continue
    # Give names the familiar M01 reference; other fields get their own readable references.
    # Separate IDs prevent one field's missingness overwriting another's log entry.
    check_id = "M01" if field == "fire_name" else "M_" + field
    table_name = "Evidence_" + check_id
    flags = missing_flags[field]
    # Keep name, state and the field being checked as context. dict.fromkeys removes a
    # repeated field name while retaining order; checking state should not insert state twice
    # in the evidence table.
    columns = list(dict.fromkeys(["fire_name", "state", field]))
    tables[table_name] = make_evidence(flags, check_id, columns)
    # Tie each absence to its intended analytical use. Name-based grouping can be limited
    # even when sizes remain usable; another field needs a priority justified by its chosen
    # question.
    missing_judgement = {
        "finding_type": "Observed absence",
        "question": f"Can the planned analysis use {field} for every record?",
        "expectation": f"The stated use requires an available {field} value.",
        "impact": f"The flagged records cannot supply {field} from the present extract.",
        "severity": "Medium" if field == "fire_name" else "Not assigned",
        "severity_reason": (
            "Name-based grouping has incomplete coverage, while other fields may remain useful."
            if field == "fire_name" else f"Judge the role of {field} in the chosen analysis before prioritising."
        ),
        "proposed_action": "Investigate source availability and define a justified analysis population.",
        "verification": "Repeat the availability check and report included/excluded records for the stated question."
    }
    # Register the count and complete evidence for this field before the next iteration
    # changes flags. Missingness populations overlap, so their issue counts must not be added
    # to claim unique affected records.
    record_check(check_id, f"Missing {field}", field, total_records, flags.sum(), table_name,
                 reference=f"Notebook 01, section 4: raw[{field!r}].isna()",
                 judgement=missing_judgement)
display(missing_summary)


### 3.3 Row completeness and overlapping absences

**Question:** Are whole records absent, and do missing names overlap with missing containment dates?
Keep summaries rather than create a second full copy of every record already supported by M01/M02.
The four overlap rows account for the whole source exactly once. M03 separately identifies records
where **every** source field is missing.


**Documentation:** [DataFrame.all](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.all.html) · [DataFrame.sum](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html) · [Series.value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) · [pd.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html).


In [ ]:
# Summarise row completeness and overlapping absences without making extra full copies of
# evidence already kept for M01 and M02.

# all(axis=1) requires every field in a source row to be missing. By contrast, sum(axis=1)
# records how many fields are missing, whether that is one, two or all of them.
entirely_missing = missing_flags.all(axis=1)
missing_per_record = missing_flags.sum(axis=1)
# Group the per-record missing-field counts. rename_axis gives the index a meaningful heading
# before reset_index converts it into an exportable column: a row labelled 2 counts records
# missing two fields.
row_summary = missing_per_record.value_counts().sort_index().rename_axis("missing_fields").reset_index(name="record_count")
tables["Missing_Row_Summary"] = row_summary
tables["Evidence_M03"] = make_evidence(entirely_missing, "M03", list(raw.columns))
# Completely absent source rows cannot contribute observed field information. This separate
# finding concerns whole-row absence; it is not a second copy of every record missing just a
# name or date.
record_check("M03", "Entirely missing records", "all fields", total_records,
             entirely_missing.sum(), "Evidence_M03",
             reference="Notebook 01, section 4.4: raw.isna().all(axis=1)",
             judgement={**m02_judgement, "finding_type": "Record without observed information",
                        "question": "Can every source record contribute information to an analysis?",
                        "expectation": "A useful source record needs at least one observed field.",
                        "impact": "The flagged records contribute no observed source-field information.",
                        "severity": "Not assigned", "severity_reason": "Investigate whether these are export artefacts and how they affect the population.",
                        "proposed_action": "Investigate the original export before agreeing a treatment.",
                        "verification": "Repeat whole-row missingness and reconcile source/retained record counts."})

missing_names = missing_flags["fire_name"]
# Cross the missing-name and missing-containment decisions at matching source labels. Each
# record falls into exactly one of four combinations; include both False/True labels even
# when a combination is absent.
overlap = pd.crosstab(missing_names, missing_containment).reindex(
    index=[False, True], columns=[False, True], fill_value=0
)
overlap_rows = []
# The nested loops enumerate all four Boolean pairs. For each pair, retrieve its count from
# the cross-tabulation and append a dictionary, giving Excel a simple four-row table.
for name_missing in [False, True]:
    for containment_missing in [False, True]:
        overlap_rows.append({
            "name_missing": name_missing, "containment_missing": containment_missing,
            "record_count": int(overlap.loc[name_missing, containment_missing])
        })
tables["Missing_Overlap"] = pd.DataFrame(overlap_rows)
# any counts each source record once if it lacks one or more fields. Adding M01 and M02
# instead would count records missing both twice.
print("Records with any missing field:", missing_flags.any(axis=1).sum())
display(tables["Missing_Overlap"])


## 4. Record exact repetitions — R01

**Question:** Which complete source rows repeat, and what do the two duplicate counts mean?
Evidence keeps **all members** of repeated groups. `later_repetition` identifies the copies after
the first occurrence. Repetition alone does not prove duplicate physical fires: this extract has
no incident identifier. Do not sum members and later copies; they describe overlapping populations.


**Documentation:** [DataFrame.duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html).


In [ ]:
# Keep every member of repeated full-row groups so partners can be reviewed, while recording
# later-copy counts separately.

# The default omits the first row in each repeated group. keep=False includes it, so three
# identical rows give two later repetitions but three review candidates.
later_repetitions = raw.duplicated()
exact_members = raw.duplicated(keep=False)
# Include original fields and a later_repetition flag. A reviewer can now see which partners
# were retained and why two duplicate-counting settings give different totals.
tables["Evidence_R01"] = make_evidence(exact_members, "R01", list(raw.columns),
                                        {"later_repetition": later_repetitions})
# All members minus later copies equals the number of repeated groups because each group
# contributes one first occurrence. These are counts of stored row values, not confirmed
# distinct physical incidents.
duplicate_summary = pd.DataFrame([
    {"comparison": "Original rows", "measure": "All repeated-group members", "count": int(exact_members.sum())},
    {"comparison": "Original rows", "measure": "Later repetitions", "count": int(later_repetitions.sum())},
    {"comparison": "Original rows", "measure": "Repeated groups", "count": int(exact_members.sum() - later_repetitions.sum())}
])
tables["Duplicate_Summary"] = duplicate_summary
# The issue concerns whether records can be treated as distinct incidents. Matching values
# without an incident identifier cannot settle identity; an identity decision is needed
# before deletion.
repetition_judgement = {
    "finding_type": "Repeated source values",
    "question": "Can source records be treated as distinct incidents when counting fires?",
    "expectation": "A distinct-incident count needs a documented record identity rule.",
    "impact": "Repeated values can affect counts, but equal rows do not establish incident identity.",
    "severity": "High",
    "severity_reason": "Distinct-incident counts require an identity decision before deletion or deduplication.",
    "proposed_action": "Investigate source identifiers and record meaning before deciding how to handle repetitions.",
    "verification": "Apply the agreed identity rule and reconcile retained records and incident counts."
}
# R01 measures all group members, so its affected count reconciles with the complete evidence
# length. Later copies stay a supporting measure rather than being mixed into the same count.
record_check("R01", "Exact repeated-group members", "all source fields", total_records,
             exact_members.sum(), "Evidence_R01",
             reference="Notebook 01, section 5.2: raw.duplicated(keep=False)",
             judgement=repetition_judgement,
             notes=f"Later repetitions: {int(later_repetitions.sum()):,}; evidence contains all group members.")
display(tables["Evidence_R01"].head(10))


## 5. Record useful categories and text representations

**Question:** Do populated labels provide the information needed for grouping?
Frequency tables show the original cause, state and size-class labels, including absent cells.
C01 checks the allowed but uninformative `Missing/Undefined` cause separately from missing cells.


**Documentation:** [Series.value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) · [Series.rename_axis](https://pandas.pydata.org/docs/reference/api/pandas.Series.rename_axis.html) · [Series.reset_index](https://pandas.pydata.org/docs/reference/api/pandas.Series.reset_index.html).


In [ ]:
# Capture categorical context and distinguish an undefined cause from an absent cause cell.

category_tables = []
# Count original values for one field at a time, including missing cells. Each frequency row
# means a field-label group, not an individual fire.
for field in ["fire_cause", "state", "fire_class"]:
    counts = raw[field].value_counts(dropna=False).rename_axis("original_value").reset_index(name="record_count")
    # Attach the field name before combining tables: an A value from fire_class must not be
    # mistaken for a cause label. concat stacks the compatible frequency rows into one export
    # table.
    counts.insert(0, "field", field)
    category_tables.append(counts)
tables["Category_Counts"] = pd.concat(category_tables, ignore_index=True)

# The literal Missing/Undefined value is present text, so it is not found by a missing-cell
# condition. Its equality mask identifies a different limitation: the known cause is
# unavailable.
undefined_cause = (raw["fire_cause"] == "Missing/Undefined").fillna(False)
tables["Evidence_C01"] = make_evidence(undefined_cause, "C01", ["fire_name", "state", "fire_cause"])
# Use all records for the question about the whole reported population. The proposed
# treatment keeps unknown causes visible or explicitly defines an exclusion; it cannot guess
# a known cause.
record_check("C01", "Undefined cause labels", "fire_cause", total_records,
             undefined_cause.sum(), "Evidence_C01",
             reference='Notebook 02, section 3.1: fire_cause == "Missing/Undefined"',
             judgement={
                 "finding_type": "Present but undefined information",
                 "question": "What proportion of all reported records has each known cause?",
                 "expectation": "A known-cause comparison needs an identified cause or an explicit undefined group.",
                 "impact": "The flagged records cannot be assigned a known cause from their label.",
                 "severity": "Medium", "severity_reason": "Known-cause comparisons require an explicit treatment of undefined records.",
                 "proposed_action": "Keep undefined cause visible and document whether each analysis includes or excludes it.",
                 "verification": "Reconcile known, undefined and genuinely missing cause counts against the stated population."
             })
# This is a shortened display of the combined context. The full category tables remain
# registered for export and later reconciliation to the source count.
display(tables["Category_Counts"].head(15))


### 5.1 Surrounding whitespace — T01 and related fields

**Question:** Could spaces split labels that we intend to compare together?
Evidence keeps both the original and trimmed values. The comparison changes temporary Series,
not the source. Check each text field separately, using only present values as the denominator.


**Documentation:** [Series.astype](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) · [Series.str.strip](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html) · [Series.str.casefold](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.casefold.html) · [Series.str.len](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.len.html).


In [ ]:
# Inspect whitespace consistently across grouping fields, preserving originals beside
# comparison values and using present labels as the denominator.

# Prepare name Series now because later placeholder, variant and character checks will reuse
# them. Nullable text preserves absent names as missing rather than inventing a literal "nan"
# label.
names = raw["fire_name"].astype("string")
trimmed_names = names.str.strip()
comparison_names = trimmed_names.str.casefold()
text_summaries = []
# Repeat the same check for name, state, class and cause. Each iteration has its own field,
# ID and table name so the evidence and denominator remain field-specific.
for field in ["fire_name", "state", "fire_class", "fire_cause"]:
    values = raw[field].astype("string")
    trimmed = values.str.strip()
    folded = trimmed.str.casefold()
    # A label must be present AND differ from its trimmed version. " CA " is flagged because
    # trimming changes it to CA; CA is unchanged. Unknown comparisons are outside this mask,
    # not certified passes.
    spaces = (values.notna() & (values != trimmed)).fillna(False)
    check_id = "T01" if field == "fire_name" else "T01_" + field
    table_name = "Evidence_" + check_id
    # Keep the original field plus trimmed_value and the length difference. " CA " has four
    # characters and CA has two, so two removed characters explain the equality-test flag.
    tables[table_name] = make_evidence(spaces, check_id, [field], {
        "trimmed_value": trimmed, "characters_removed": values.str.len() - trimmed.str.len()
    })
    # Distinct counts show whether formatting created extra groups, which is different from
    # counting changed records. Fifty padded values can change many records while merging
    # only one label choice.
    text_summaries.append({
        "field": field, "present_count": int(values.notna().sum()),
        "whitespace_count": int(spaces.sum()), "distinct_original": int(values.nunique()),
        "distinct_trimmed": int(trimmed.nunique()), "distinct_case_comparison": int(folded.nunique())
    })
    # Only present values could be compared for spelling, so use their count as the
    # denominator. A separate cleaned dataset and agreed text policy are proposed; this
    # comparison edits no source values.
    record_check(check_id, f"Surrounding whitespace in {field}", field, values.notna().sum(),
                 spaces.sum(), table_name,
                 reference=f"Notebook 02, section 4: present {field} differs after strip()",
                 judgement={
                     "finding_type": "Text-format difference",
                     "question": f"Can records be grouped consistently by {field}?",
                     "expectation": "Surrounding whitespace should not create unintended groups.",
                     "impact": "Equivalent labels may appear in separate groups.",
                     "severity": "Medium", "severity_reason": "The stated grouping may change when surrounding whitespace is compared consistently.",
                     "proposed_action": "Review the evidence and agree a text policy for a separate cleaned dataset.",
                     "verification": "Repeat the whitespace check, explain distinct-label changes and verify unchanged record counts."
                 })
tables["Text_Summary"] = pd.DataFrame(text_summaries)
display(tables["Text_Summary"])


### 5.2 Blank and possible placeholder names — T02

**Question:** Can a present name still lack useful information?
Whitespace-only names and candidate placeholders are checked separately. A word such as
`unknown` is a screening candidate whose meaning needs investigation. It is not automatically missing.

**Name-screen examples:** A present `"   "` becomes empty and is flagged as whitespace-only.
`" UNKNOWN "` becomes comparison value `unknown` and is a placeholder candidate.
A missing name is outside both present-name screens and remains recorded by M01.
The candidate list makes the test reproducible without deciding its meaning automatically.


**Documentation:** [Series.fillna](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) · [Series.isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html).


In [ ]:
# Screen two kinds of potentially uninformative present names without merging them with
# ordinary missing-cell findings.

present_names = names.notna()
# A whitespace-only string becomes empty after trimming; a missing string remains outside
# this comparison. The placeholder screen uses a separate candidate list because a word such
# as unknown is not automatically an absent name.
blank_names = (trimmed_names == "").fillna(False)
possible_placeholders = comparison_names.isin(placeholder_words)
# Priority stays unassigned while meaning is uncertain. The source definition must decide
# whether a candidate is a legitimate label or unavailable information.
name_judgement = {
    "finding_type": "Possible uninformative label",
    "question": "Do recorded names provide meaningful labels for name-based analysis?",
    "expectation": "A useful name label should be distinguishable from an absent or undefined name.",
    "impact": "Uninformative labels may combine unrelated records or overstate named-record coverage.",
    "severity": "Not assigned", "severity_reason": "Confirm the meaning and intended use of the candidate names first.",
    "proposed_action": "Review original records and agree which labels, if any, represent unavailable names.",
    "verification": "Repeat the agreed classification and reconcile present, absent and undefined name counts."
}
# Each tuple supplies an ID, description, computed decision Series and written check
# reference. The two iterations preserve separate counts even though they share the
# evidence-building pattern.
for check_id, title, flags, rule in [
    ("T02_blank", "Whitespace-only names", blank_names, 'trimmed_names == ""'),
    ("T02", "Possible placeholder names", possible_placeholders, "comparison_names.isin(placeholder_words)")
]:
    table_name = "Evidence_" + check_id
    # Keep original names and comparison names together so a reviewer can see why " UNKNOWN "
    # matched a candidate. This is full flagged evidence, not a sample.
    tables[table_name] = make_evidence(flags, check_id, ["fire_name", "state"], {"comparison_name": comparison_names})
    # Both screens inspect present names, so neither should use missing names to inflate its
    # denominator. An observed candidate creates a review issue, not a claim that cleaning is
    # already justified.
    record_check(check_id, title, "fire_name", present_names.sum(), flags.sum(), table_name,
                 reference=f"Notebook 02, section 3.3: {rule}", judgement=name_judgement)


### 5.3 Distinct spelling variants — T04

**Question:** Which distinct original names share a trimmed, case-insensitive comparison value?
One row here means one distinct original spelling, not one affected fire record. Repeated identical
spellings are collapsed. We retain the first source-row example and the full occurrence count for
each spelling. Similar names can still refer to different fires.


**Documentation:** [DataFrame.dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) · [DataFrame.drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) · [Series.duplicated](https://pandas.pydata.org/docs/reference/api/pandas.Series.duplicated.html) · [Series.map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html).


In [ ]:
# Find formatting variants among DISTINCT spellings while retaining a source example and
# frequency for each spelling.

# Pair each original name with its own comparison name and source label. Exclude absent names
# from the comparison; they remain present as records in raw.
name_pairs = pd.DataFrame({
    "example_source_row": raw.index, "original_name": names, "comparison_name": comparison_names
}).dropna(subset=["original_name", "comparison_name"])
# Compare only original_name and comparison_name when collapsing pairs. Including
# example_source_row would keep every occurrence distinct and incorrectly treat frequency as
# spelling variation.
name_pairs = name_pairs.drop_duplicates(subset=["original_name", "comparison_name"])
# After identical pairs are collapsed, a repeated comparison value means multiple distinct
# originals share it. keep=False includes every spelling in each such group.
several_versions = name_pairs["comparison_name"].duplicated(keep=False)
name_variants = name_pairs.loc[several_versions].sort_values("comparison_name").copy()
name_variants.insert(0, "issue_id", "T04")
# Map each original spelling to its full frequency in names. This separates one variant row
# from the many records that use that spelling; the retained source label is only its first
# example.
name_variants["source_record_count"] = name_variants["original_name"].map(names.value_counts())
tables["Name_Variants"] = name_variants
# Use distinct spellings as both checked and affected units. A variant-row count is not an
# affected-fire count, and equivalent names still do not establish incident identity.
record_check("T04", "Distinct spellings sharing a comparison name", "fire_name", len(name_pairs),
             len(name_variants), "Name_Variants", unit="distinct spellings",
             reference="Notebook 02, section 4.4b: deduplicate original/comparison pairs, then flag repeated comparison values",
             judgement={**name_judgement,
                        "finding_type": "Name-format variants",
                        "impact": "A name-based grouping can split formatting variants or merge unrelated same-name records.",
                        "proposed_action": "Agree a comparison policy and investigate identity before using names as grouping keys."},
             evidence_scope="All flagged distinct spellings; first source-row example and complete frequency per spelling")
display(name_variants.head(10))


### 5.4 Additional repetitions after text comparison — R02

**Question:** Does consistent text comparison reveal additional repeated-row candidates?
R02 counts records newly joining repeated groups. Its evidence also includes their matching
partners, which may already have been in an exact-repeat group. `newly_flagged` identifies the
rows counted for R02, so the evidence can contain more rows than the affected count.

**Why evidence can exceed the affected count:** Suppose all other fields match in three rows:
two original names are `"Oak"`, and the third is `" Oak "`. The first two were already an exact
repeated group. After trimming, the third joins them, so R02 counts **one newly flagged record**
but keeps **all three partners** as evidence. Its reconciliation counts the `newly_flagged`
True values, not the full evidence-table length.


**Documentation:** [DataFrame.copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [DataFrame.duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) · [MultiIndex.from_frame](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_frame.html) · [Index.isin](https://pandas.pydata.org/docs/reference/api/pandas.Index.isin.html).


In [ ]:
# R02 counts newly flagged records, but useful evidence must also show all their matching
# partners under the temporary text comparison.

# Use a separate table and compare selected label fields after trimming and casefolding. All
# other source values still participate unchanged in full-row matches.
comparison = raw.copy()
for field in ["fire_name", "state", "fire_class", "fire_cause"]:
    comparison[field] = raw[field].astype("string").str.strip().str.casefold()
comparison_members = comparison.duplicated(keep=False)
# Require comparison-group membership AND NOT previous exact-group membership. If two
# identical Oak rows already matched and a padded " Oak " row now joins them, only that third
# row is newly flagged.
newly_flagged = comparison_members & ~exact_members

# Retrieve whole candidate groups only when new flags exist. With no new candidates, an
# all-False source-indexed mask keeps an empty evidence table rather than inventing groups.
if newly_flagged.any():
    # Treat each complete comparison row as a tuple of its field values. The MultiIndex
    # supplies those whole-row keys so matching uses all columns, not a name alone.
    all_keys = pd.MultiIndex.from_frame(comparison)
    new_keys = pd.MultiIndex.from_frame(comparison.loc[newly_flagged])
    # Membership in the new-row keys selects EVERY source row with a matching tuple,
    # including previously flagged partners. In the Oak example the evidence has three rows,
    # but only one newly_flagged=True.
    full_candidate_groups = pd.Series(all_keys.isin(new_keys), index=raw.index)
    # The Boolean selection now retains the required group membership, so the large temporary
    # keys can be released without losing the evidence selection.
    del all_keys, new_keys
else:
    full_candidate_groups = pd.Series(False, index=raw.index)
# Keep the new-member flag and comparison text beside original fields. This lets the reviewer
# distinguish rows counted for R02 from partners retained to explain their matches.
comparisons = {"newly_flagged": newly_flagged}
for field in ["fire_name", "state", "fire_class", "fire_cause"]:
    comparisons["comparison_" + field] = comparison[field]
tables["Evidence_R02"] = make_evidence(full_candidate_groups, "R02", list(raw.columns), comparisons)
# Additional later copies are a separate measure: two unique rows that become a pair add one
# later copy but two newly flagged members. Keep that distinction in the supporting summary.
extra_later = int(comparison.duplicated().sum() - later_repetitions.sum())
tables["Duplicate_Summary"] = pd.concat([
    tables["Duplicate_Summary"], pd.DataFrame([
        {"comparison": "Trimmed/case-folded text", "measure": "All repeated-group members", "count": int(comparison_members.sum())},
        {"comparison": "Trimmed/case-folded text", "measure": "Newly flagged members", "count": int(newly_flagged.sum())},
        {"comparison": "Trimmed/case-folded text", "measure": "Additional later repetitions", "count": extra_later}
    ])
], ignore_index=True)
# Reconcile using the newly_flagged column because the complete group evidence can have more
# rows than the affected count. Counting its full length would misstate the R02 finding.
record_check("R02", "Newly flagged repeated-row candidates", "all fields; four text fields compared",
             total_records, newly_flagged.sum(), "Evidence_R02",
             reference="Notebook 02, section 5: comparison.duplicated(keep=False) & ~exact_members",
             judgement={**repetition_judgement, "finding_type": "Additional row-match candidates",
                        "severity": "Not assigned", "severity_reason": "Formatting equivalence does not establish incident identity."},
             evidence_scope="Complete newly affected comparison groups, including already flagged partners",
             notes=f"Count the True newly_flagged evidence rows; additional later repetitions: {extra_later:,}.",
             reconcile="newly_flagged")

# Release the large temporary comparison after its original and derived evidence values have
# been copied into registered tables. raw still contains all source rows.
del comparison, comparisons


### 5.5 Control characters and non-ASCII names — T03

**Question:** Do names contain characters that deserve review?
Non-ASCII includes legitimate accented letters. Control characters can affect display or export.
Keep the original name and a quoted representation that exposes hidden characters.


**Documentation:** [Series.str.contains](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html) · [Series.map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html).


In [ ]:
# Separate control-character concerns from non-ASCII names, and retain representations that
# help a reviewer see the actual supplied text.

# Controls such as tabs and newlines can be invisible or affect export. Non-ASCII letters
# such as é may be legitimate parts of names, so the second pattern is a review screen rather
# than an invalid-name rule.
control_characters = names.str.contains(r"[\x00-\x1f\x7f]", na=False)
non_ascii = names.str.contains(r"[^\x00-\x7f]", na=False)
# map applies the small lambda to each name: present values get repr with quotes and visible
# escapes; absent values get None instead of a invented textual name. This representation
# supplements the original, not replaces it.
shown_names = names.map(lambda value: repr(value) if pd.notna(value) else None)
# Run the two screens separately because the same source can contain legitimate accents
# without containing problematic controls. Each has its own evidence and count.
for check_id, title, flags in [
    ("T03_control", "Names with ASCII control characters", control_characters),
    ("T03", "Names with non-ASCII characters", non_ascii)
]:
    table_name = "Evidence_" + check_id
    tables[table_name] = make_evidence(flags, check_id, ["fire_name", "state"], {"name_repr": shown_names})
    # Only present names can be checked for characters. The written action requests source
    # and encoding review; a match alone cannot justify deleting an accent or a record.
    record_check(check_id, title, "fire_name", present_names.sum(), flags.sum(), table_name,
                 reference="Notebook 02, section 6: control-character and non-ASCII regular-expression screens",
                 judgement={
                     "finding_type": "Characters to review",
                     "question": "Are the recorded names represented and displayed accurately?",
                     "expectation": "Preserve legitimate names and distinguish them from encoding/export artefacts.",
                     "impact": "Some values may display differently or need special handling during export.",
                     "severity": "Not assigned", "severity_reason": "A flagged character alone does not establish an error or analytical impact.",
                     "proposed_action": "Review original values, encoding and source context; retain legitimate characters.",
                     "verification": "Compare retained values with the source and repeat the agreed character checks."
                 })


## 6. Test numeric and date representations — P01/P02

**Question:** Can present sizes and years be read as usable numbers?
An absent input is already covered by missingness. A present input that becomes missing during
conversion is a conversion failure. Infinity converts successfully but is checked separately.
Evidence retains the source text and the temporary converted value.

**Numeric rule examples:** `"12.5"` converts successfully; present `"unknown"` becomes missing
and is a conversion failure; an originally absent input is a completeness finding;
`"inf"` converts to infinity and needs its own non-finite representation check.
Successful conversion alone cannot validate an area or reported year.


**Documentation:** [pd.to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) · [np.isinf](https://numpy.org/doc/stable/reference/generated/numpy.isinf.html) · [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html).


In [ ]:
# Establish numerical representations before checking meaning, with separate evidence for
# failed conversions and converted infinities.

# Coercion creates temporary numeric results without replacing source values. A present
# "unknown" becomes missing; an absent input was already missing; "inf" can convert to
# infinity.
size_numbers = pd.to_numeric(raw["fire_size"], errors="coerce")
year_numbers = pd.to_numeric(raw["fire_year"], errors="coerce")
# Wrap the finite-number decisions with the original source index so later eligibility masks
# select the correct records. Unknown decisions are excluded, and representation findings
# remain separate.
finite_size = pd.Series(np.isfinite(size_numbers), index=raw.index).fillna(False)
finite_year = pd.Series(np.isfinite(year_numbers), index=raw.index).fillna(False)

# Reuse the analytical requirement for a usable representation, while leaving priority to the
# affected calculation. The source value must be reviewed before a correction or exclusion is
# agreed.
representation_judgement = {
    "finding_type": "Unusable representation",
    "question": "Can present source values support the intended numerical or date calculation?",
    "expectation": "A present value must convert to the required usable type.",
    "impact": "Flagged values cannot safely enter the intended calculation as supplied.",
    "severity": "Not assigned", "severity_reason": "Assess the role and population of the affected calculation first.",
    "proposed_action": "Review the original values and source definition before agreeing a correction or exclusion.",
    "verification": "Repeat conversion and eligibility checks and reconcile usable, absent and failed values."
}
# Each tuple ties a short reference label to a source field and its numeric result, letting
# the same logic inspect size and year without mixing their values.
for short_name, field, numbers in [
    ("size", "fire_size", size_numbers), ("year", "fire_year", year_numbers)
]:
    present = raw[field].notna()
    # A conversion failure needs original presence AND a missing numeric result. An
    # originally absent value belongs to its missingness check, not this failure population.
    conversion_failed = present & numbers.isna()
    # Infinity needs its own test because it can be a non-missing successful numeric
    # conversion. It is still unsuitable for ordinary year and acre measurements.
    infinite = pd.Series(np.isinf(numbers), index=raw.index).fillna(False)
    # Use present inputs as the conversion denominator, but converted non-missing results as
    # the infinity-test denominator. Eligibility must match what each screen could actually
    # inspect.
    for suffix, title, flags, eligible in [
        ("parse", f"Numeric conversion failures in {field}", conversion_failed, present),
        ("infinite", f"Infinite numeric values in {field}", infinite, numbers.notna())
    ]:
        check_id = f"P01_{short_name}_{suffix}"
        table_name = "Evidence_" + check_id
        # Retain the original and converted number together so the evidence demonstrates the
        # failure or infinity rather than merely reporting a flag.
        tables[table_name] = make_evidence(flags, check_id, [field], {"converted_number": numbers})
        record_check(check_id, title, field, eligible.sum(), flags.sum(), table_name,
                     reference=f"Notebook 02, section 7.2: {suffix} rule on present/converted {field}",
                     judgement=representation_judgement)


### 6.1 Test date conversion and text shape separately

**Question:** Is each present date both formatted as expected and a usable calendar value?
A string can have the right shape but contain an impossible month. Conversely, a format difference
needs investigation even if a converter can interpret it. Missing dates are excluded from both
checks and remain in the missingness log. Temporary converted dates support later relationship checks.

**Date rules can give different results:** `2001-99-19 00:00:00` matches the expected digit and
separator shape but fails calendar conversion because month 99 is impossible. A missing date
cannot be checked as present text and is recorded by missingness instead. The two date screens
may overlap on a record, so their issue counts are not additive.


**Documentation:** [pd.to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) · [Series.str.fullmatch](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.fullmatch.html).


In [ ]:
# Inspect calendar interpretation and written timestamp layout separately, then retain usable
# dates for later relationships.

# Parse into separate source-indexed Series using the declared format. Impossible or absent
# timestamps become NaT, while the original source strings remain available.
discovery_dates = pd.to_datetime(raw["discovery_date"], format=DATE_FORMAT, errors="coerce")
contained_dates = pd.to_datetime(raw["contained_date"], format=DATE_FORMAT, errors="coerce")
date_summary_rows = []
# Associate each source endpoint with its own converted Series. Reusing the steps for both
# dates makes their summaries comparable.
for short_name, field, dates in [
    ("discovery", "discovery_date", discovery_dates),
    ("contained", "contained_date", contained_dates)
]:
    present = raw[field].notna()
    date_text = raw[field].astype("string")
    # Only a present source value that becomes missing is a failure. Wrong shape separately
    # means present text does not match the full digit/separator pattern.
    failed_parse = present & dates.isna()
    # 2001-99-19 00:00:00 has the expected shape but cannot supply a calendar date. These two
    # tests therefore need separate references and counts; their flagged populations can
    # overlap.
    wrong_shape = present & ~date_text.str.fullmatch(TIMESTAMP_PATTERN, na=False)
    # Keep present, usable, failed and shape counts together with the readable date range. A
    # zero failure count among present values does not restore absent endpoints.
    date_summary_rows.append({
        "field": field, "present_count": int(present.sum()), "usable_date_count": int(dates.notna().sum()),
        "parse_failure_count": int(failed_parse.sum()), "shape_failure_count": int(wrong_shape.sum()),
        "earliest_usable_date": dates.min(), "latest_usable_date": dates.max()
    })
    # One loop iteration records calendar conversion and the other records text layout. Both
    # use present values as the checked population and retain original/converted evidence.
    for suffix, title, flags in [
        ("parse", f"Date conversion failures in {field}", failed_parse),
        ("shape", f"Unexpected date-text shape in {field}", wrong_shape)
    ]:
        check_id = f"P02_{short_name}_{suffix}"
        table_name = "Evidence_" + check_id
        tables[table_name] = make_evidence(flags, check_id, [field], {"converted_date": dates})
        record_check(check_id, title, field, present.sum(), flags.sum(), table_name,
                     reference=f"Notebook 02, section 8: explicit {DATE_FORMAT} conversion and timestamp-shape screen",
                     judgement=representation_judgement)
# Combine the per-endpoint dictionaries into one exportable context table. Each row now
# describes a date FIELD, not a source fire.
tables["Date_Summary"] = pd.DataFrame(date_summary_rows)
display(tables["Date_Summary"])


## 7. Validate categories and numerical meaning

**Question:** Do present category labels match the documented allowed labels?
The allowed lists were supplied before inspection. We do not use the observed labels as their
own acceptance rule. Keep category membership separate from the two-uppercase-letter state format.
A correctly shaped `ZZ` would still fail jurisdiction membership.

**Membership versus format rule:** `CA` meets both expectations. `ZZ` fits two uppercase letters
but is absent from the allowed jurisdiction list. `ca` fails the exact documented spelling and
uppercase format. A missing code supplies no present text to inspect. An allowed cause such as
`Missing/Undefined` passes membership while still lacking a known cause for C01.


**Documentation:** [Series.isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) · [Series.str.fullmatch](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.fullmatch.html).


In [ ]:
# Validate source categories against independently declared references, then test state-code
# shape as a separate question.

# An unexpected label might be formatting, a new documented category or an error. Record that
# uncertainty rather than guessing a mapping from the label alone.
category_judgement = {
    "finding_type": "Value outside a documented category rule",
    "question": "Can the recorded category support the intended grouping?",
    "expectation": "Present labels should follow the documented category or code rule.",
    "impact": "Unexpected labels may create unintended groups or prevent valid comparisons.",
    "severity": "Not assigned", "severity_reason": "Investigate whether this is formatting, a new valid category or a source error.",
    "proposed_action": "Compare the original label with the source documentation and agree a mapping only when justified.",
    "verification": "Repeat the agreed category/format rule and reconcile original and final group counts."
}
category_reference_rows = []
# Each tuple chooses the field and its corresponding allowed list. State membership must use
# state codes, while fire_class membership uses A-G.
for check_id, field, allowed in [
    ("K01", "fire_class", allowed_classes), ("K02", "state", allowed_states),
    ("K03", "fire_cause", allowed_causes)
]:
    present = raw[field].notna()
    # A present label outside the independent list is flagged. CA passes jurisdiction
    # membership and ZZ does not; missing values remain untested for membership.
    invalid = present & ~raw[field].isin(allowed)
    table_name = "Evidence_" + check_id
    tables[table_name] = make_evidence(invalid, check_id, [field])
    record_check(check_id, f"Invalid documented labels in {field}", field, present.sum(),
                 invalid.sum(), table_name, reference="Notebook 03, section 3: present value not in independent allowed list",
                 judgement=category_judgement)
    # The reference summary includes every allowed label. get(label, 0) explicitly records
    # zero when an allowed label never appears, distinguishing coverage from supplied invalid
    # labels.
    observed_counts = raw[field].value_counts()
    for label in allowed:
        category_reference_rows.append({"field": field, "allowed_label": label,
                                        "record_count": int(observed_counts.get(label, 0))})
tables["Category_Reference"] = pd.DataFrame(category_reference_rows)

state_text = raw["state"].astype("string")
# Two uppercase letters are the shape requirement: CA and ZZ fit, ca and CAA do not. ZZ can
# therefore pass K04 shape while failing K02 membership.
bad_state_shape = state_text.notna() & ~state_text.str.fullmatch(r"[A-Z]{2}", na=False)
tables["Evidence_K04"] = make_evidence(bad_state_shape, "K04", ["state"])
# Record shape separately so one pass cannot be mistaken for complete code validity. Its
# evidence uses originals, with absent codes excluded from the denominator.
record_check("K04", "Unexpected jurisdiction-code format", "state", state_text.notna().sum(),
             bad_state_shape.sum(), "Evidence_K04", reference="Notebook 03, section 3.3: fullmatch('[A-Z]{2}')",
             judgement=category_judgement)


### 7.1 Validate whole years, the extract's year range and positive sizes

**Question:** Are the converted numbers meaningful for these fields?
Only finite values are eligible. Fractional years and years outside 1992–2015 are independent
checks, so their findings may overlap. Positive size is needed before class and distribution checks.

**Numeric meaning examples:** `1998.5` is finite and inside the period but fails whole-year
meaning; `2020` is finite and whole but outside the period. A value such as `2020.5` can fail
both independent year checks. For sizes, finite `0` and `-2` are flagged, finite `0.1` meets the
positive-size rule, and unavailable/non-finite sizes remain untested for that meaning rule.


**Documentation:** [Series.where](https://pandas.pydata.org/docs/reference/api/pandas.Series.where.html) · [Series.between](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html).


In [ ]:
# Successful conversion is not the final verdict: check whole years, the extract period and
# positive sizes before later comparisons.

# where leaves non-finite years missing in this temporary remainder calculation, so infinity
# does not enter it. Remainder % 1 is zero for 1998 and 0.5 for 1998.5; the latter is
# flagged.
fractional_year = finite_year & (year_numbers.where(finite_year) % 1 != 0)
# Use the independently expected year endpoints, inclusive. 1991 or 2016 is outside this
# extract; 1998.5 is inside the interval but still fails the whole-year test.
year_outside_range = finite_year & ~year_numbers.between(min(expected_years), max(expected_years))
# A finite 0 or -2 acres violates positive size. A finite 0.1 passes this condition; a
# missing or infinite size cannot enter this meaning check.
non_positive_size = finite_size & (size_numbers <= 0)
# Require ALL year prerequisites together so a whole out-of-range value or an in-range
# fractional value cannot enter valid annual grouping. valid_size will similarly restrict
# class and distribution checks.
valid_year = (finite_year & ~fractional_year & ~year_outside_range).fillna(False)
valid_size = (finite_size & (size_numbers > 0)).fillna(False)
# These field-specific rules explain why a readable number may still be unsuitable for
# analysis. Changing a source value needs an investigation, not just a failing comparison.
range_judgement = {
    "finding_type": "Value outside a documented numeric rule",
    "question": "Can reported years and sizes support valid annual and size analysis?",
    "expectation": "Years are whole values in 1992–2015; measured sizes are positive finite acres.",
    "impact": "Flagged values can distort annual groups or size calculations.",
    "severity": "Not assigned", "severity_reason": "Assess the affected analysis and verify source meaning first.",
    "proposed_action": "Investigate original values and source definitions before choosing a correction or exclusion.",
    "verification": "Repeat the numerical rule and reconcile eligible and excluded record counts."
}
# Keep the three checks distinct because fractional and out-of-period findings can overlap on
# one record. Each percentage uses its finite eligible population and retains the converted
# value beside the source.
for check_id, title, field, flags, eligible, numbers in [
    ("N01", "Fractional reported years", "fire_year", fractional_year, finite_year, year_numbers),
    ("N02", "Years outside the extract range", "fire_year", year_outside_range, finite_year, year_numbers),
    ("N03", "Zero or negative fire sizes", "fire_size", non_positive_size, finite_size, size_numbers)
]:
    table_name = "Evidence_" + check_id
    tables[table_name] = make_evidence(flags, check_id, [field], {"converted_number": numbers})
    record_check(check_id, title, field, eligible.sum(), flags.sum(), table_name,
                 reference="Notebook 03, section 4: whole/range year checks or finite size <= 0",
                 judgement=range_judgement)


## 8. Record coverage and annual changes — V01/V02

**Question:** Where does the extract have no eligible records, and which annual changes deserve context?
Coverage uses valid reported years and allowed jurisdictions. A zero count means no eligible
records in this extract for that combination. It does not prove that no physical fires occurred.
Evidence for V01 is a full jurisdiction–year table, including zero-count combinations.


**Documentation:** [Series.value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) · [Series.reindex](https://pandas.pydata.org/docs/reference/api/pandas.Series.reindex.html) · [pd.crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) · [DataFrame.reindex](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reindex.html) · [DataFrame.stack](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.stack.html).


In [ ]:
# Show all expected groups and their counts so gaps remain visible, using group combinations
# rather than source records as the coverage-check unit.

# A valid reported year AND an allowed jurisdiction define comparable source records. Use
# converted whole-year values in a separate table to keep grouping labels consistent.
coverage_eligible = valid_year & raw["state"].isin(allowed_states)
coverage_data = raw.loc[coverage_eligible, ["fire_year", "state"]].copy()
coverage_data["fire_year"] = year_numbers.loc[coverage_eligible].astype("int64")
# Reindex the frequency result to the complete expected calendar list. This both restores
# chronological order and records absent years as zero, supporting annual comparisons later.
year_counts = coverage_data["fire_year"].value_counts().reindex(expected_years, fill_value=0)
coverage_years = year_counts.rename_axis("fire_year").reset_index(name="record_count")
coverage_years["empty_year"] = coverage_years["record_count"] == 0
tables["Coverage_Years"] = coverage_years

# Count each record by its pair of jurisdiction and year. Seeing CA somewhere and 2000
# somewhere is not enough to establish a CA-2000 record.
state_year_counts = pd.crosstab(coverage_data["state"], coverage_data["fire_year"])
# Build the full allowed-jurisdiction by expected-year grid, including combinations with zero
# observations. These are the combinations whose reporting coverage may need explanation.
state_year_counts = state_year_counts.reindex(index=allowed_states, columns=expected_years, fill_value=0)
state_year_counts.index.name = "state"
state_year_counts.columns.name = "fire_year"
# stack turns the grid into one row per jurisdiction-year pair, and reset_index makes the two
# grouping labels explicit columns. Excel can now filter and inspect each pair directly.
coverage_combinations = state_year_counts.stack().rename("record_count").reset_index()
coverage_combinations.insert(0, "issue_id", "V01")
# A zero is an observation about eligible records in this extract, not proof of zero physical
# fires. The flag requests coverage context.
coverage_combinations["empty_combination"] = coverage_combinations["record_count"] == 0
tables["Coverage_State_Year"] = coverage_combinations

coverage_judgement = {
    "finding_type": "Extract coverage to investigate",
    "question": "Can reported records be compared consistently across the expected jurisdictions and years?",
    "expectation": "Establish the reporting scope of each jurisdiction–year combination before comparing activity.",
    "impact": "Zero extract counts cannot distinguish no reported activity from unavailable reporting coverage.",
    "severity": "Not assigned", "severity_reason": "The source reporting scope is needed to interpret these zero counts.",
    "proposed_action": "Check reporting coverage and distinguish unsupported comparisons from observed zero counts.",
    "verification": "Document coverage for each questioned combination and reconcile counts for the agreed analysis scope."
}
# The complete grid includes non-empty pairs, so reconcile by the empty_combination flag. Its
# checked count is pairs; eligible source-record counts belong in the notes as supporting
# context.
record_check("V01", "Empty jurisdiction-year combinations", "state / fire_year", len(coverage_combinations),
             coverage_combinations["empty_combination"].sum(), "Coverage_State_Year",
             unit="jurisdiction-year combinations", reference="Notebook 03, section 5.3: complete expected state/year grid with zero counts",
             judgement=coverage_judgement, evidence_scope="All expected combinations and record counts",
             notes=f"Eligible source records: {int(coverage_eligible.sum()):,}; excluded source records: {int((~coverage_eligible).sum()):,}.",
             reconcile="empty_combination")

# Register the all-years summary as a supporting check without adding a duplicate issue. It
# provides another view of the same coverage investigation.
record_check("V01_year", "Years with no eligible records", "fire_year", len(coverage_years),
             coverage_years["empty_year"].sum(), "Coverage_Years", unit="years",
             reference="Notebook 03, section 5.1: expected year_counts == 0",
             evidence_scope="All expected years and record counts", reconcile="empty_year",
             notes="Supporting coverage summary; consider with V01.")


### 8.1 Compare each year with the previous year

**Question:** Which calculable annual changes meet the 25% review threshold?
The first year has no predecessor. A previous count of zero cannot be a percentage denominator.
These cases are untested for percentage change, even when their review flag is False.

**Annual-change rule examples:** 100 → 125 records is +25% and meets the threshold; 100 → 75
is -25% and also meets it; 100 → 124 is +24% and does not. A previous zero or no preceding year
makes the percentage comparison untested. The threshold is a classroom review choice, not
a limit on how much true fire activity may change.


**Documentation:** [Series.shift](https://pandas.pydata.org/docs/reference/api/pandas.Series.shift.html) · [Series.replace](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html) · [Series.abs](https://pandas.pydata.org/docs/reference/api/pandas.Series.abs.html).


In [ ]:
# Compare neighbouring calendar years with a defined denominator and retain both tested and
# untested annual comparisons.

# Chronological order was established above. shift places last year's count under the current
# year label, so current and previous counts align for subtraction.
previous_counts = year_counts.shift(1)
# The first year has no predecessor, and a previous zero cannot support division. Leave their
# percentage change unavailable rather than interpreting them as stable years.
safe_previous_counts = previous_counts.replace(0, np.nan)
# A change from 100 to 130 gives +30%; from 100 to 70 gives -30%. The denominator is the
# PREVIOUS count, not the current count or the whole-file record count.
year_changes = (year_counts - previous_counts) / safe_previous_counts * 100
change_eligible = year_changes.notna()
# Require a calculable change and use its absolute size so rises and falls both meet the 25%
# review choice. Exact +25% and -25% meet the inclusive threshold.
large_change = change_eligible & (year_changes.abs() >= ANNUAL_CHANGE_THRESHOLD)
# All these arrays follow the same expected-year order. Keep the eligibility column because
# False review flags with missing change values mean untested, not passed.
year_change_report = pd.DataFrame({
    "fire_year": year_counts.index, "record_count": year_counts.to_numpy(),
    "previous_record_count": previous_counts.to_numpy(), "change_percent": year_changes.to_numpy(),
    "change_eligible": change_eligible.to_numpy(), "review_flag": large_change.to_numpy()
})
year_change_report.insert(0, "issue_id", "V02")
tables["Coverage_Year_Changes"] = year_change_report
# Count annual comparisons, not fires. The evidence retains every expected year, so reconcile
# only its True review_flag rows; the first-year exclusion remains visible in checked versus
# total counts.
record_check("V02", "Annual changes meeting the review threshold", "annual eligible record counts",
             change_eligible.sum(), large_change.sum(), "Coverage_Year_Changes", unit="annual comparisons",
             total_count=len(expected_years), reference="Notebook 03, section 5.2: absolute annual percentage change >= 25",
             judgement={**coverage_judgement,
                        "finding_type": "Annual count change to investigate",
                        "question": "How should changes in annual reported-record counts be interpreted?",
                        "impact": "Large changes can reflect fire activity, reporting changes or extract coverage.",
                        "proposed_action": "Seek activity and reporting context before interpreting a threshold flag as a data problem."},
             evidence_scope="All expected years; change_eligible and review_flag identify tested and flagged comparisons",
             reconcile="review_flag")
display(year_change_report.loc[year_change_report["review_flag"]])


## 9. Record relationships between values — L01/P03/P04

**Question:** Do reported years agree with discovery years for records eligible for both checks?
L01 requires a valid reported year and a usable discovery date. Investigate definitions before
deciding which field should govern annual analysis. The severity remains Not assigned until that
question and its affected population have been evaluated; no four-record reference result is assumed.

**L01 example:** A valid reported year of 1993 with discovery in 1994 is flagged because those
two fields place the same record in different annual groups. With no usable discovery date,
the record cannot enter this comparison. The source definition decides how an observed
disagreement should be treated; the inequality condition alone does not.


**Documentation:** [Series.dt.year](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.year.html).


In [ ]:
# L01 checks whether two usable within-record year values support consistent annual grouping;
# it does not decide which definition is authoritative.

# Extract the year from each interpreted discovery date at its original source label.
# fire_year=1993 and a discovery date in 1994 give two different possible annual groups.
discovery_year = discovery_dates.dt.year
# Require a valid reported year and a readable date before comparing. An absent or failed
# date cannot establish agreement or disagreement.
year_can_be_checked = valid_year & discovery_dates.notna()
# Combine the prerequisite mask with inequality, and exclude unknown flags. The affected
# population therefore contains actual eligible comparisons that disagree.
year_mismatch = (year_can_be_checked & (year_numbers != discovery_year)).fillna(False)
# Keep original year/date values beside the derived discovery year. That makes the exact
# disagreement reproducible instead of expecting the reviewer to trust a count alone.
tables["Evidence_L01"] = make_evidence(year_mismatch, "L01", [
    "fire_name", "state", "fire_year", "discovery_date", "contained_date"
], {"discovery_year": discovery_year})
# Individually usable fields can still disagree as a pair. Shared judgement keeps that
# concern distinct from a claim that one field is definitely wrong.
relationship_judgement = {
    "finding_type": "Within-record relationship to investigate",
    "question": "Can the compared fields support a consistent analysis?",
    "expectation": "The relationship must agree with the field definitions and the chosen analysis question.",
    "impact": "Disagreements can alter grouping or derived calculations.",
    "severity": "Not assigned", "severity_reason": "Investigate source definitions and the affected analysis population first.",
    "proposed_action": "Review the original records and definitions before choosing a correction or accepted difference.",
    "verification": "Repeat the relationship rule after the decision and reconcile changed or accepted records."
}
# Tailor the shared judgement to discovery-based annual analysis. Leave severity unassigned
# until source definitions and the actual affected population justify a priority.
record_check("L01", "Reported/discovery year disagreements", "fire_year / discovery_date",
             year_can_be_checked.sum(), year_mismatch.sum(), "Evidence_L01",
             reference="Notebook 03, section 6.1: valid reported year differs from usable discovery year",
             judgement={**relationship_judgement,
                        "question": "Can annual record counts use a consistent discovery year?",
                        "expectation": "Confirm whether fire_year is intended to agree with the discovery year.",
                        "impact": "The same record can enter different annual groups depending on which year field is selected.",
                        "verification": "Repeat the year comparison and verify that annual groups follow the agreed year definition."})


### 9.1 Check date order — P03

**Question:** Is containment earlier than discovery when both dates are usable?
Records with absent or failed dates are not checked for date order. Evidence shows the original
values beside converted dates so that the comparison can be reproduced.

**Order rule example:** Discovery 10 July/containment 9 July is flagged; containment 12 July
is not. Equal timestamps do not violate this earlier-than check. Missing either usable endpoint
means untested, so zero failures cannot certify every source record for duration analysis.


In [ ]:
# P03 tests whether two interpreted endpoints can provide a non-negative
# discovery-to-containment duration.

# Both usable dates are prerequisites. A record missing containment belongs to its
# availability finding and is untested for order, even when discovery is valid.
both_dates_present = discovery_dates.notna() & contained_dates.notna()
# Discovery 10 July and containment 9 July is flagged because subtraction would give a
# negative duration. Equal endpoints are not earlier and pass this order condition.
contained_too_early = both_dates_present & (contained_dates < discovery_dates)
# Retain original strings and converted endpoints together so another person can repeat the
# same chronological comparison.
tables["Evidence_P03"] = make_evidence(contained_too_early, "P03", [
    "fire_name", "discovery_date", "contained_date"
], {"converted_discovery": discovery_dates, "converted_containment": contained_dates})
# Use the both-endpoints population as the denominator. Zero failures applies only to that
# population; records with unavailable endpoints did not pass this test.
record_check("P03", "Containment before discovery", "discovery_date / contained_date",
             both_dates_present.sum(), contained_too_early.sum(), "Evidence_P03",
             reference="Notebook 03, section 6.2: both usable dates and containment < discovery",
             judgement={**relationship_judgement,
                        "question": "Can a non-negative discovery-to-containment duration be calculated?",
                        "expectation": "Containment should not precede discovery for this duration definition.",
                        "impact": "Flagged records would produce negative durations."})


### 9.2 Build the expected fire-size class, then compare it — P04

**Question:** Does the reported class match the positive finite size under the same boundaries
used in Notebook 03? A = (0, 0.25], B = (0.25, 10), C = [10, 100), D = [100, 300),
E = [300, 1,000), F = [1,000, 5,000), G = [5,000, infinity).
Only records with a valid size and an allowed reported class are eligible for comparison.

**Boundary examples:** 0.25 acres belongs to `A`; 0.26 to `B`; 10 to `C`; 100 to `D`;
300 to `E`; 1,000 to `F`; and 5,000 to `G`. Thus 10 acres reported as `B` is flagged,
while 10 reported as `C` agrees. The brackets in the intervals above include an endpoint;
parentheses exclude it. A missing size or an undocumented reported `H` cannot enter P04.


**Documentation:** [pd.Series](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [Series.loc](https://pandas.pydata.org/docs/reference/api/pandas.Series.loc.html) · [Series.isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html).


In [ ]:
# P04 derives a size-based expectation independently of the reported class, then compares the
# two only when both are usable.

# Start with a missing expectation at every source label. Invalid sizes should keep an
# unavailable expectation instead of receiving a misleading default class.
expected_class = pd.Series(pd.NA, index=raw.index, dtype="string")
# Positive 0.25 acres belongs to A; 0.26 belongs to B. valid_size already excludes zero,
# negative, missing and infinite sizes.
expected_class.loc[valid_size & (size_numbers <= 0.25)] = "A"
expected_class.loc[valid_size & (size_numbers > 0.25) & (size_numbers < 10)] = "B"
# Inclusive lower and exclusive upper endpoints assign 10 -> C, 100 -> D, 300 -> E, 1000 -> F
# and 5000 -> G. The boundary choices stop a value being put into two classes or none.
expected_class.loc[valid_size & (size_numbers >= 10) & (size_numbers < 100)] = "C"
expected_class.loc[valid_size & (size_numbers >= 100) & (size_numbers < 300)] = "D"
expected_class.loc[valid_size & (size_numbers >= 300) & (size_numbers < 1000)] = "E"
expected_class.loc[valid_size & (size_numbers >= 1000) & (size_numbers < 5000)] = "F"
expected_class.loc[valid_size & (size_numbers >= 5000)] = "G"
# An allowed A-G reported label is also required. Missing classes or H are separate
# category/completeness findings rather than size-class disagreements.
class_can_be_checked = valid_size & raw["fire_class"].isin(allowed_classes)
# Size 10 with reported B is flagged because its expectation is C. The source index keeps the
# reported and expected classes attached to the same record.
class_mismatch = (class_can_be_checked & (raw["fire_class"] != expected_class)).fillna(False)
# Put size, reported class and derived expectation together so the reason for each
# disagreement is directly inspectable. The expected Series does not replace the source
# class.
tables["Evidence_P04"] = make_evidence(class_mismatch, "P04", ["fire_name", "fire_size", "fire_class"],
                                        {"converted_size": size_numbers, "expected_class": expected_class})
record_check("P04", "Reported/expected size-class disagreements", "fire_size / fire_class",
             class_can_be_checked.sum(), class_mismatch.sum(), "Evidence_P04",
             reference="Notebook 03, section 7: class expected from positive finite size differs from allowed reported class",
             judgement={**relationship_judgement,
                        "question": "Can comparisons by fire-size class use a consistent classification?",
                        "expectation": "Reported class should match the documented size intervals.",
                        "impact": "Affected records can appear in the wrong size-class group."})


## 10. Record statistical review flags — O01

**Question:** Which positive finite sizes fall outside the 1.5-IQR fences, and how might they affect
average-size analysis? Keep the boundaries and distribution summary so that the screen can be
reproduced. A large real fire can be a valid outlier. With no eligible sizes the check is Untested.

**Statistical rule example:** With illustrative Q1=1 acre and Q3=5 acres, IQR=4 and the
1.5-IQR fences are -5 and 11 acres. A valid 12-acre value is flagged; exactly 11 is not.
The actual boundaries are computed below and stored with the evidence context. A flag calls
for plausibility and impact review rather than automatic correction or deletion.


**Documentation:** [Series.quantile](https://pandas.pydata.org/docs/reference/api/pandas.Series.quantile.html) · [Series.describe](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html).


In [ ]:
# Use an explicitly recorded distribution screen to find sizes deserving review, while
# keeping the source-defined positive-size rule separate.

# Only positive finite sizes can contribute to these quartiles and comparisons. Keep the
# source labels so flagged measurements can be traced back to their records.
sizes = size_numbers.loc[valid_size]
# Q1 and Q3 bound the middle half of the eligible size distribution. IQR is their difference,
# and the multiplier extends that spread into review fences.
q1 = sizes.quantile(0.25)
q3 = sizes.quantile(0.75)
iqr = q3 - q1
# With illustrative Q1=1, Q3=5 and multiplier=1.5, the fences are -5 and 11 acres. These are
# computed review boundaries, not universal minimum and maximum valid wildfire sizes.
lower_fence = q1 - IQR_MULTIPLIER * iqr
upper_fence = q3 + IQR_MULTIPLIER * iqr
# A valid size below the lower OR above the upper fence is flagged. Strict comparisons leave
# an exactly-on-fence value unflagged. Keep full source-indexed decisions for evidence
# selection.
unusual_size = (valid_size & ((size_numbers < lower_fence) | (size_numbers > upper_fence))).fillna(False)
tables["Evidence_O01"] = make_evidence(unusual_size, "O01", ["fire_name", "state", "fire_size", "fire_class"],
                                        {"converted_size": size_numbers})
# Retain quartiles and upper-tail percentiles so the distribution screen can be reproduced
# and interpreted. rename_axis and reset_index turn statistic labels into a normal export
# column.
size_statistics = sizes.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).rename_axis("statistic").reset_index(name="value")
# The count statistic measures records; size summaries measure acres. The list comprehension
# chooses the unit for each statistic so numbers of different meanings are not presented as
# interchangeable.
size_statistics["unit"] = ["records" if value == "count" else "acres" for value in size_statistics["statistic"]]
# Store the multiplier and both calculated fences alongside the distribution. Without these,
# an outlier count would not tell another person which screen produced it.
fence_statistics = pd.DataFrame([
    {"statistic": "IQR", "value": iqr, "unit": "acres"},
    {"statistic": "IQR multiplier", "value": IQR_MULTIPLIER, "unit": "multiplier"},
    {"statistic": "Lower fence", "value": lower_fence, "unit": "acres"},
    {"statistic": "Upper fence", "value": upper_fence, "unit": "acres"}
])
tables["Size_Statistics"] = pd.concat([size_statistics, fence_statistics], ignore_index=True)
# An empty eligible population is Untested, and a valid extreme fire may still be flagged.
# Source plausibility and analytical effect must be reviewed before any deletion or
# correction.
record_check("O01", "Fire sizes beyond the IQR fences", "fire_size", len(sizes), unusual_size.sum(), "Evidence_O01",
             reference="Notebook 03, section 8: positive finite size outside Q1 - 1.5*IQR or Q3 + 1.5*IQR",
             judgement={
                 "finding_type": "Statistical review flag",
                 "question": "How do unusually large or small fires affect average fire size?",
                 "expectation": "Review values beyond the IQR fences in their source and analytical context.",
                 "impact": "Extreme sizes can influence the mean and comparisons between groups.",
                 "severity": "Not assigned", "severity_reason": "An IQR flag alone does not show whether a value is wrong or materially affects the chosen analysis.",
                 "proposed_action": "Investigate source plausibility and distinguish confirmed errors from valid extreme fires.",
                 "verification": "Repeat the screen with recorded boundaries and document the decisions on reviewed values."
             }, notes="Fence boundaries and all eligible-size summary statistics are retained in Size_Statistics.")
display(tables["Size_Statistics"])


## 11. Assemble and review the complete logs

**Question:** Does every recorded finding have reproducible evidence and a justified next step?
These DataFrames combine the dictionaries accumulated above. No issue is invented for a passed
check. The check log still retains zero results, eligible counts and untested populations.

The written judgements are worked starting points. Review them for your analysis and edit the
relevant judgement dictionary above, then restart and run the notebook. Workbook regeneration
replaces the generated workbook, so manual Excel edits should first be transferred back here.


**Documentation:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [DataFrame.sort_values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html).


In [ ]:
# Convert the accumulated dictionaries into reviewable tables only after each check has
# supplied its measured facts and any justified issue.

# One dictionary becomes one issue row. Fixed headings also keep an empty issue table valid;
# sorting references and resetting its display index do not alter the separately retained
# source references in evidence.
issue_log = pd.DataFrame(issue_entries, columns=ISSUE_COLUMNS).sort_values("issue_id").reset_index(drop=True)
# All checks remain here, including passes and untested populations. A short issue list
# therefore does not mean only a few inspections were performed.
check_log = pd.DataFrame(check_entries, columns=CHECK_COLUMNS).sort_values("check_id").reset_index(drop=True)
# Register the completed log DataFrames with the other export tables. Later evidence-sheet
# names are added to these same objects, so the workbook gets the final linked logs.
tables["Issue_Log"] = issue_log
tables["Check_Log"] = check_log
print("Issue entries:", len(issue_log))
print("Checks recorded:", len(check_log))
# Show a compact view for review, including count_unit. The full tables still contain impact,
# expectation, verification and other written answers for export.
display(issue_log[["issue_id", "finding_type", "affected_count", "checked_count", "count_unit", "severity"]])
display(check_log[["check_id", "flagged_count", "checked_count", "not_checked_count", "count_unit", "outcome"]])


### 11.1 Validate counts, evidence and source preservation

**Question:** Do full evidence populations agree with the log, using the correct unit and rule?
Most checks count evidence rows. R02 counts True `newly_flagged` rows, and complete summary
evidence counts its particular Boolean flag column. We also reconcile summaries to the source
population and verify that the source DataFrame was not changed.

**What “correct unit and rule” means:** For M02, two evidence rows mean two affected records.
For a five-row schema report with two True `flagged` values, the affected count is two column
labels. For the three-partner R02 example above, only one True `newly_flagged` value contributes
to its affected count. These different evidence lengths are all valid when their counting
instructions match the finding. This validation rule checks consistency, not whether a source
value is substantively correct.


**Documentation:** [Series.is_unique](https://pandas.pydata.org/docs/reference/api/pandas.Series.is_unique.html) · [pd.util.hash_pandas_object](https://pandas.pydata.org/docs/reference/api/pandas.util.hash_pandas_object.html) · [Series.equals](https://pandas.pydata.org/docs/reference/api/pandas.Series.equals.html).


In [ ]:
# Before saving, reconcile the recorded counts to the retained evidence and confirm that the
# inspection has preserved its source.

# An assert checks an expected invariant and stops execution if it is false. Unique
# issue/check references prevent ambiguous links; shape plus row-content hashes check more
# than row counts alone.
assert issue_log["issue_id"].is_unique, "Issue references must be unique."
assert check_log["check_id"].is_unique, "Check references must be unique."
assert raw.shape == original_shape, "The source shape changed."
assert pd.util.hash_pandas_object(raw, index=True).equals(original_content_hash), "The source values changed."

# Find each check's registered evidence and its counting instruction. Most evidence counts
# rows; a full summary or complete R02 group counts True values in a specified flag column.
for check in check_entries:
    table = tables[check["evidence_table"]]
    rule = evidence_rules[check["check_id"]]
    # For a five-row summary with two flags, evidence_count is 2, not 5. R02 similarly counts
    # newly_flagged=True while retaining extra partners as explanatory evidence.
    evidence_count = len(table) if rule == "rows" else int(table[rule].sum())
    assert evidence_count == check["flagged_count"], f"{check['check_id']}: evidence/count disagreement."
    # Checked plus untested must reconstruct the recorded total in the SAME count unit. Zero
    # checked inputs must have no percentage and cannot claim a Passed outcome.
    assert check["not_checked_count"] + check["checked_count"] == check["total_count"]
    if check["checked_count"] == 0:
        assert check["percent"] is None and check["outcome"] in {"Untested", "Not applicable"}

# Match each issue back to its check by the shared reference. The affected and checked counts
# must agree rather than reflecting independently edited copies.
for entry in issue_entries:
    matching_check = next(check for check in check_entries if check["check_id"] == entry["issue_id"])
    assert entry["affected_count"] == matching_check["flagged_count"]
    assert entry["checked_count"] == matching_check["checked_count"]
    assert entry["severity"] in {"Low", "Medium", "High", "Not assigned"}
    # isinstance verifies written answers are text; strip then requires meaningful content
    # rather than spaces only. These checks detect missing answers but cannot judge whether
    # the reasoning is correct.
    for field in ["question", "expectation", "impact", "severity_reason", "proposed_action", "verification"]:
        assert isinstance(entry[field], str) and entry[field].strip(), f"{entry['issue_id']}: missing {field}."

# The four overlap groups and missing-field summary each partition the whole source, so their
# counts should each sum to total_records. This would fail if a group were accidentally
# omitted or counted twice.
assert int(tables["Missing_Overlap"]["record_count"].sum()) == total_records
assert int(tables["Missing_Row_Summary"]["record_count"].sum()) == total_records
# Frequency tables include missing values, so every source record must contribute once per
# field. Coverage tables instead sum to eligible source records, because their prerequisites
# excluded some inputs.
for field in ["fire_cause", "state", "fire_class"]:
    assert int(tables["Category_Counts"].loc[tables["Category_Counts"]["field"] == field, "record_count"].sum()) == total_records
assert int(tables["Coverage_Years"]["record_count"].sum()) == int(coverage_eligible.sum())
assert int(tables["Coverage_State_Year"]["record_count"].sum()) == int(coverage_eligible.sum())
# The complete grid needs one row for every allowed-jurisdiction and expected-year pair.
# Multiplying the list lengths checks coverage of the report structure, not whether every
# pair has actual fires.
assert len(tables["Coverage_State_Year"]) == len(allowed_states) * len(expected_years)
print("Validation passed: log counts, complete evidence, summaries and unchanged source agree.")


## 12. Prepare one workbook with a sheet for every table

**Question:** Where will each log and its evidence be saved?
[Excel permits 1,048,576 worksheet rows](https://support.microsoft.com/en-us/excel/excel-specifications-and-limits?app=wp).
We reserve one heading row, leaving **1,048,575 data rows**. A larger table is split into numbered
sheets in the same workbook. `Sheet_Index` lists every exported log/evidence partition, its complete
table size and its zero-based positions within that table. `evidence_sheets` in both logs gives
the actual worksheet names. Empty tables still receive a heading-only sheet.

The default `auto` export uses the familiar Pandas writer for smaller workbooks. Above 200,000
cells, it uses [openpyxl write-only mode](https://openpyxl.readthedocs.io/en/stable/optimized.html)
to avoid keeping millions of Excel cell objects in memory. Both paths export the same complete
DataFrames, apply the same layout and create **one** workbook. The full wildfire evidence may
take several minutes to write; progress is printed for each sheet. No sampling or truncation occurs.


In [ ]:
# Set the export capacity and provide workbook context so someone opening the Excel file can
# interpret the logs without this notebook.

# Excel permits 1,048,576 worksheet rows; reserve one for headings, leaving 1,048,575 data
# rows. Large evidence must be partitioned, not truncated.
EXCEL_DATA_ROW_LIMIT = 1_048_575
MAX_DATA_ROWS = EXCEL_DATA_ROW_LIMIT
# auto chooses a memory-saving writer for many cells and the familiar Pandas writer for
# smaller outputs. The threshold changes implementation, not which evidence is retained.
EXPORT_METHOD = "auto"
STREAM_CELL_THRESHOLD = 200_000

# Pair a readable heading with its explanation or measured run value. Fingerprint,
# environment, units and screening choices let a reviewer understand the result's provenance
# and limits.
guide_rows = [
    ("Module", "PRT2030"), ("Source file", DATASET_PATH.name),
    ("Source SHA-256", source_sha256), ("Run time (UTC)", run_time_utc),
    ("Source records", total_records), ("Source columns", len(raw.columns)),
    ("Python used", sys.version.split()[0]), ("Pandas used", pd.__version__),
    ("NumPy used", np.__version__), ("openpyxl used", openpyxl.__version__),
    ("Teaching target", "Python 3.14; Pandas 3.0.5"),
    ("Record meaning", "One reported wildfire record. The extract has no incident identifier."),
    ("Source reference", "source_row is the zero-based label of the unchanged Pandas import, not a physical CSV line or incident ID."),
    ("Issue log", "One issue per row; passed checks appear in Check_Log without invented issues."),
    ("Check outcomes", "Passed; Flagged for review; Untested; Not applicable. Passed applies only to checked items."),
    ("Count units", "Use count_unit: records, columns, distinct spellings, years or jurisdiction-year combinations."),
    ("Percentage scale", "Percent and change_percent store 0-100 percentage values, not fractional rates."),
    ("Evidence scope", "Complete flagged evidence or a complete labelled summary; preview displays are never exported as full evidence."),
    ("R02 evidence", "Contains complete newly affected matching groups. Count newly_flagged=True, not every evidence row."),
    ("Name variants", "One distinct spelling per row, with its first example_source_row and full source_record_count."),
    ("Coverage", "Zero records for a combination indicate extract coverage, not proof of zero physical fires."),
    ("IQR screen", "A review flag does not prove a measurement is wrong. Boundaries are in Size_Statistics."),
    ("Severity", "Low, Medium, High or Not assigned. Review written judgements against the stated question."),
    ("Progress", "Open - decision required is a proposed investigation/treatment, not completed cleaning."),
    ("Repeated counts", "Finding counts overlap. Do not add them to claim a unique problem-record count."),
    ("Additional fields", "Required fields are checked first; any extra source fields are retained and logged for review."),
    ("Allowed jurisdiction codes", " ".join(allowed_states)),
    ("Allowed size classes", ", ".join(allowed_classes)),
    ("Allowed causes", "; ".join(allowed_causes)),
    ("Expected years", "1992-2015 inclusive"), ("Date format", DATE_FORMAT),
    ("Placeholder screen", ", ".join(placeholder_words)),
    ("Annual-change review threshold (%)", ANNUAL_CHANGE_THRESHOLD),
    ("IQR multiplier", IQR_MULTIPLIER),
    ("Class boundaries (acres)", "A: (0,0.25]; B: (0.25,10); C: [10,100); D: [100,300); E: [300,1000); F: [1000,5000); G: [5000,infinity)."),
    ("Regeneration", "Regenerating replaces this workbook after a successful write. Keep reusable edits in the notebook."),
    ("Text in Excel", "Literal text remains text, including names beginning with '='. XML-prohibited controls are shown as visible hexadecimal escape sequences in Excel only; in-memory evidence is unchanged."),
    ("Unavailable numeric values", "Missing values are blank in Excel; positive/negative infinity is shown as inf/-inf text."),
    ("Early converted dates", "Converted dates before 1900 are shown as ISO text because Excel's usual date system starts at 1900. Original source dates remain available."),
    ("Pandas export documentation", "https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html"),
    ("Large-file export documentation", "https://openpyxl.readthedocs.io/en/stable/optimized.html"),
    ("Excel worksheet limits", "https://support.microsoft.com/en-us/excel/excel-specifications-and-limits?app=wp")
]
# Each tuple becomes one guide row with item and value headings. Unlike issue entries, these
# rows document the workbook and do not count source findings.
tables["Workbook_Guide"] = pd.DataFrame(guide_rows, columns=["item", "value"])

# List context reports explicitly so they appear in a predictable order. Then place detailed
# Evidence_ tables after the guide, logs and summaries.
summary_names = [
    "Schema_Report", "Header_Report", "Column_Summary", "Missing_Summary",
    "Missing_Row_Summary", "Missing_Overlap", "Category_Counts", "Category_Reference",
    "Text_Summary", "Name_Variants", "Duplicate_Summary", "Date_Summary",
    "Coverage_Years", "Coverage_State_Year", "Coverage_Year_Changes", "Size_Statistics"
]

ordered_names = ["Workbook_Guide", "Issue_Log", "Check_Log"] + summary_names
# Discover registered evidence by its shared prefix so adding a legitimate evidence table
# does not silently exclude it from export.
ordered_names += sorted(name for name in tables if name.startswith("Evidence_"))
# The exported name set must cover all registered tables. An unlisted table would otherwise
# vanish from the workbook even though its check existed in memory.
assert set(ordered_names) == set(tables), "An unregistered table would be omitted from export."


### 12.1 Plan partitions and link the logs to their actual sheets

This helper keeps sheet names valid and unique, including Excel's 31-character name limit.
Splitting uses table positions; it keeps every data row once and repeats headings for each part.
This export preparation does not change source values or inspection results.

**Partition example:** With eight evidence rows and an illustrative capacity of three data rows
per sheet, export positions 0–2, 3–5 and 6–7 into three sheets. Each gets its own heading row.
Every data row appears once. These are positions in the evidence table, not the potentially
non-consecutive `source_row` labels saved inside it. The real default capacity is much larger.


**Documentation:** [DataFrame.iloc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) · [Series.map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html).


In [ ]:
# Plan every worksheet and link logical evidence tables to their actual sheet names before
# either export method writes data.

# A partition must hold at least one data row and must fit under the worksheet data limit.
# Reject an invalid setting before constructing incomplete export plans.
if not 1 <= MAX_DATA_ROWS <= EXCEL_DATA_ROW_LIMIT:
    raise ValueError("MAX_DATA_ROWS must be between 1 and 1,048,575.")
# Keep previously assigned names so later truncation or sanitisation cannot give two
# different tables the same worksheet name.
used_sheet_names = set()

def unique_sheet_name(table_name, suffix=""):

    # Replace Excel-forbidden punctuation, remove edge apostrophes and provide a fallback for
    # an empty name. Leave space inside the 31-character limit for partition suffixes.
    base = re.sub(r"[:\\/?*\[\]]", "_", table_name).strip("'") or "Table"
    candidate = base[:31 - len(suffix)] + suffix
    number = 1
    # Two long names can become identical after truncation, and Excel compares names without
    # case distinctions. Try numbered endings until the candidate is unique, then remember
    # it.
    while candidate.casefold() in used_sheet_names:
        extra = f"_{number}"
        candidate = base[:31 - len(suffix) - len(extra)] + suffix + extra
        number += 1
    used_sheet_names.add(candidate.casefold())
    return candidate

def plan_table(table_name):
    table = tables[table_name]

    # Round the table-size division upward so a partly filled final sheet is included. With
    # eight rows and capacity three we need three parts; max(1, ...) gives an empty evidence
    # table a heading-only sheet.
    number_of_parts = max(1, math.ceil(len(table) / MAX_DATA_ROWS))
    result = []
    for part in range(number_of_parts):
        # The example parts use table positions [0:3], [3:6] and [6:8]. The stop is
        # exclusive, so these slices cover every row exactly once with no overlap. Positions
        # here are not the original source_row labels.
        start = part * MAX_DATA_ROWS
        stop = min(start + MAX_DATA_ROWS, len(table))
        # Only split tables need numbered suffixes such as _001. Store both the logical table
        # name and physical sheet name so evidence references remain understandable.
        suffix = f"_{part + 1:03d}" if number_of_parts > 1 else ""
        result.append({
            "table_name": table_name, "sheet_name": unique_sheet_name(table_name, suffix),
            "part": part + 1, "start": start, "stop": stop,
            "data_rows": stop - start, "table_rows": len(table)
        })
    return result

export_plan = []
for table_name in ordered_names:
    # Each table can produce several part dictionaries. extend adds all of those planned
    # sheets to one ordered export list rather than nesting a list inside it.
    export_plan.extend(plan_table(table_name))

sheet_lookup = {}
for part in export_plan:
    # Build a mapping from a logical table to all its actual sheets. setdefault starts an
    # empty list the first time that table is encountered, then appends each partition name.
    sheet_lookup.setdefault(part["table_name"], []).append(part["sheet_name"])

# Map each logged evidence table to its sheet-name list and join the names for Excel. This
# adds navigable references without replacing the original logical table reference.
issue_log["evidence_sheets"] = issue_log["evidence_table"].map(lambda name: "; ".join(sheet_lookup[name]))
check_log["evidence_sheets"] = check_log["evidence_table"].map(lambda name: "; ".join(sheet_lookup[name]))

index_rows = []
for part in export_plan:
    # Record each partition's inclusive first/last table positions and row counts. The last
    # position is stop-1 because the slice stop itself was excluded; an empty part has no
    # first or last data row.
    index_rows.append({
        "table_name": part["table_name"], "sheet_name": part["sheet_name"], "part": part["part"],
        "first_table_row": part["start"] if part["data_rows"] else None,
        "last_table_row": part["stop"] - 1 if part["data_rows"] else None,
        "data_rows": part["data_rows"], "complete_table_rows": part["table_rows"]
    })
tables["Sheet_Index"] = pd.DataFrame(index_rows)

# Plan the index itself after its rows are known. Put its sheet after the guide; its contents
# describe the other planned tables rather than recursively listing itself.
index_parts = plan_table("Sheet_Index")
guide_parts = [part for part in export_plan if part["table_name"] == "Workbook_Guide"]
other_parts = [part for part in export_plan if part["table_name"] != "Workbook_Guide"]
export_plan = guide_parts + index_parts + other_parts

# Estimate workbook size using each part's data rows plus one heading row, times its column
# count. Even empty tables need heading cells.
total_export_cells = sum((part["data_rows"] + 1) * len(tables[part["table_name"]].columns) for part in export_plan)
if EXPORT_METHOD not in {"auto", "pandas", "stream"}:
    raise ValueError("EXPORT_METHOD must be auto, pandas or stream.")
# With auto, compare that estimate to the streaming threshold. An explicit pandas or stream
# choice overrides the automatic selection; all paths use the same export plan.
selected_method = (
    "stream" if total_export_cells > STREAM_CELL_THRESHOLD else "pandas"
) if EXPORT_METHOD == "auto" else EXPORT_METHOD
print("Sheets planned:", len(export_plan))
print("Export method:", selected_method)
print("All evidence rows are retained.")
display(tables["Sheet_Index"].head(12))


### 12.2 Shared Excel values and layout

The next functions prepare **Excel copies** and apply headings, filters, frozen references,
number formats and readable descriptions. They do not clean the source or alter evidence in memory.
XML-prohibited control characters are displayed as visible `\xNN` sequences in Excel. Their
quoted original representation is also available in character evidence. Text beginning with `=`
is stored as literal text. A value too long for an Excel cell causes an explicit error instead of
being silently shortened.

[openpyxl styles](https://openpyxl.readthedocs.io/en/stable/styles.html) and
[worksheet filters](https://openpyxl.readthedocs.io/en/stable/filters.html) document this formatting.


In [ ]:
# Prepare Excel-compatible copies and readable layout without modifying the stored source or
# in-memory evidence.

from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.cell import WriteOnlyCell

# Define styles once and reuse them across both writers so the same logs remain recognisable
# whichever export method is selected.
header_font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
body_font = Font(name="Arial", size=10, color="222222")
header_fill = PatternFill("solid", fgColor="24445C")
header_alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
text_alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
number_alignment = Alignment(horizontal="right", vertical="center")
# Excel files contain XML, which cannot store some controls such as a zero byte. Identify
# precisely those controls for visible escaping rather than removing legitimate accented
# names.
illegal_xml = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

def excel_value(value):

    # An unavailable scalar becomes a blank Excel cell. All values passed here are individual
    # cell values, not Series or nested containers.
    if pd.isna(value):
        return None
    # A NumPy scalar may need conversion to an ordinary Python value for the writer. item
    # extracts that scalar value without changing its meaning.
    if isinstance(value, np.generic):
        value = value.item()
    # Infinity cannot be stored as an ordinary finite Excel measurement. Keep inf or -inf
    # text to make its presence explicit rather than silently converting it to zero or blank.
    if isinstance(value, float) and math.isinf(value):
        return "inf" if value > 0 else "-inf"
    # Convert a Pandas timestamp to a Python datetime for Excel. Dates before 1900 are
    # written as ISO text because the usual worksheet date system cannot represent them
    # normally.
    if isinstance(value, pd.Timestamp):
        value = value.to_pydatetime()
    if isinstance(value, datetime) and value.year < 1900:
        return value.isoformat(sep=" ")
    if isinstance(value, str):
        # Each prohibited control is displayed as a visible hexadecimal escape such as \x00.
        # This conversion belongs to the Excel copy; the evidence in memory still preserves
        # the original value.
        value = illegal_xml.sub(lambda match: "\\x" + format(ord(match.group()), "02x"), value)
        # Raise an error instead of silently shortening evidence that would exceed Excel's
        # cell-text capacity. The output must not look complete if content has been lost.
        if len(value) > 32_767:
            raise ValueError("An evidence value exceeds Excel's 32,767-character cell limit; it was not truncated.")
    return value

# Choose width by field purpose: identifiers need room for long references and narrative
# answers need wrapping space. The fallback bounds heading-based widths so an unusual field
# does not make a sheet unusably narrow or wide.
def column_width(field):
    if field in {"issue_id", "check_id"}:
        return 28
    if field in {"finding_type", "count_unit", "item"}:
        return 34
    if field in {"question", "expectation", "impact", "severity_reason", "proposed_action", "verification", "notes", "check_reference", "evidence_scope", "finding", "value"}:
        return 64
    if field in {"evidence_sheets", "evidence_table"}:
        return 42
    if field in {"original_name", "fire_name", "trimmed_value", "comparison_name", "name_repr"}:
        return 34
    if "date" in field or "converted_discovery" in field or "converted_containment" in field:
        return 26
    return max(16, min(38, len(field) + 3))

def number_format(field):

    # The notebook already stores 30 for 30%. Excel's normal percentage format would multiply
    # 30 again and show 3000%, so append a literal % sign. Annual changes can be negative or
    # exceed 100; the stored unit remains percentage.
    if "percent" in field:
        return '0.0000"%"'
    # A calendar year should display as 1998 rather than 1,998. Record counts and source
    # references use whole-number formats; dates and sizes need their own readable formats.
    if field in {"fire_year", "discovery_year"}:
        return "0"
    if field in {"source_row", "example_source_row", "part", "first_table_row", "last_table_row", "complete_table_rows", "data_rows", "missing_fields", "count"} or "count" in field:
        return "#,##0"
    if "date" in field or field in {"converted_discovery", "converted_containment"}:
        return "yyyy-mm-dd hh:mm:ss"
    if field in {"fire_size", "converted_size", "converted_number"}:
        return "#,##0.0###"
    return "General"

def configure_sheet(sheet, frame):
    sheet.sheet_view.showGridLines = False
    sheet.sheet_format.defaultRowHeight = 18
    # Freeze the heading row and reference columns so a reader scrolling far into evidence
    # can still identify the record and column meanings. C2 keeps two reference columns; B2
    # keeps one.
    sheet.freeze_panes = "C2" if frame.columns[0] in {"issue_id", "check_id"} else "B2"
    # Include headings plus every data row in the filter range. Excel can then filter the
    # whole part rather than a shortened notebook preview.
    sheet.auto_filter.ref = f"A1:{get_column_letter(len(frame.columns))}{len(frame) + 1}"
    sheet.row_dimensions[1].height = 34
    for position, field in enumerate(frame.columns, start=1):
        sheet.column_dimensions[get_column_letter(position)].width = column_width(field)

    # Fit narrative row heights only for small tables. Allocating row-specific styles for
    # millions of evidence rows would defeat the memory-saving export.
    if len(frame) <= 10_000:
        for row_number, values in enumerate(frame.itertuples(index=False, name=None), start=2):
            line_count = 1
            # Pair each row value with its heading to estimate how many wrapped lines it
            # needs. Use the longest cell in the row, and cap the resulting height at Excel's
            # practical row-height limit.
            for field, value in zip(frame.columns, values):
                if isinstance(value, str):
                    chars_per_line = max(10, int(column_width(field) - 3))
                    lines = sum(max(1, math.ceil(len(line) / chars_per_line)) for line in value.split("\n"))
                    line_count = max(line_count, lines)
            sheet.row_dimensions[row_number].height = min(409, max(18, line_count * 14 + 5))

def style_body_cell(cell, field, value, large_evidence=False):
    cell.font = body_font
    if isinstance(value, str):
        # Explicit text storage keeps a name beginning with = as literal source data instead
        # of letting Excel interpret it as a formula. This preserves the supplied name's
        # meaning.
        cell.data_type = "s"
        cell.alignment = (
            Alignment(horizontal="left", vertical="center", wrap_text=False)
            if large_evidence else text_alignment
        )
    else:
        cell.alignment = number_alignment
    cell.number_format = number_format(field)
    # A fractional-year failure such as 1998.5 must remain visibly fractional. Override the
    # normal whole-year display so formatting does not conceal the very problem the evidence
    # records.
    if field in {"fire_year", "discovery_year"} and isinstance(value, float) and not value.is_integer():
        cell.number_format = "0.0###"


### 12.3 The familiar Pandas export

This is the main pattern for smaller assignments: one writer stays open while every DataFrame
is written to its own sheet. The `with` block closes and saves the workbook after all sheets are
written. `index=False` omits the redundant DataFrame index; the explicit `source_row` remains.
The helper also handles planned sheet splits and formats the workbook.


**Documentation:** [pd.ExcelWriter](https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html) · [DataFrame.to_excel](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_excel.html) · [Series.map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html).


In [ ]:
# Use one Pandas writer for the whole workbook so every planned log and evidence part becomes
# a sheet in the same file.

def export_with_pandas(path):
    # The with block keeps one writer open for all parts and closes/saves it afterward.
    # Opening a new writer per table could replace earlier sheets instead of building one
    # combined workbook.
    with pd.ExcelWriter(path, engine="openpyxl", datetime_format="yyyy-mm-dd hh:mm:ss") as writer:
        for part in export_plan:
            # Slice by the planned start and exclusive stop positions. The copied part
            # receives Excel-only conversions, leaving the registered table and its original
            # evidence values unchanged.
            original = tables[part["table_name"]].iloc[part["start"]:part["stop"]]
            excel_copy = original.copy()
            for field in excel_copy.columns:
                # map passes each scalar to the shared conversion helper. Handling missing
                # values, infinities and prohibited controls here prepares the spreadsheet
                # without cleaning raw.
                excel_copy[field] = excel_copy[field].map(excel_value)
            # Use the unique planned sheet name and omit the redundant index. Explicit
            # source_row columns remain in evidence, so traceability does not depend on
            # writing the index again.
            excel_copy.to_excel(writer, sheet_name=part["sheet_name"], index=False)
            # Retrieve the worksheet just written to apply the shared layout. The writer owns
            # that sheet inside this one workbook.
            sheet = writer.sheets[part["sheet_name"]]
            configure_sheet(sheet, excel_copy)
            # Style headings separately from values to make the first row recognisable. Then
            # pair each body cell with its field heading so number and text formats match its
            # meaning.
            for cell in sheet[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = header_alignment
            for row in sheet.iter_rows(min_row=2):
                for field, cell in zip(excel_copy.columns, row):
                    # Large evidence uses a simpler text layout to keep export manageable. A
                    # progress message reports each complete part rather than pretending a
                    # preview is the saved evidence.
                    style_body_cell(cell, field, cell.value, len(excel_copy) > 10_000)
            print(f"Written {part['sheet_name']}: {len(excel_copy):,} data rows")


### 12.4 The memory-saving export for the full wildfire dataset

This optional technical section writes the same tables one row at a time using
[Workbook(write_only=True)](https://openpyxl.readthedocs.io/en/stable/optimized.html#write-only-mode).
It keeps the **full evidence** while avoiding the ordinary writer's large collection of Excel
cell objects. The final auto export selects this method when needed. You can use the previous
Pandas pattern for smaller assessment datasets.


**Documentation:** [DataFrame.itertuples](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.itertuples.html).


In [ ]:
# For a large workbook, write the same planned tables one row at a time so Excel cell objects
# do not accumulate for the entire dataset.

def export_with_streaming(path):
    # write_only mode keeps workbook output sequential. The original evidence DataFrames are
    # still complete in memory; this saves the extra collection of ordinary worksheet cell
    # objects.
    workbook = openpyxl.Workbook(write_only=True)
    for part in export_plan:
        # Take the same positional part used by the Pandas path, ensuring that switching
        # writers does not change the exported population or partitions.
        frame = tables[part["table_name"]].iloc[part["start"]:part["stop"]]
        sheet = workbook.create_sheet(part["sheet_name"])

        # Configure layout before appending: streamed rows cannot later be freely revisited
        # like ordinary worksheet cells.
        configure_sheet(sheet, frame)
        # Create one row of styled heading cells and append it once. WriteOnlyCell attaches a
        # value and styles to a cell before it is sent to the sheet.
        heading_cells = []
        for field in frame.columns:
            cell = WriteOnlyCell(sheet, value=field)
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = header_alignment
            heading_cells.append(cell)
        sheet.append(heading_cells)
        written_count = 0
        # itertuples supplies one row's values at a time. index=False avoids a duplicate
        # row-reference column; name=None returns simple tuples for pairing with the column
        # headings.
        for values in frame.itertuples(index=False, name=None):
            row_cells = []
            # Convert and format each value using its own heading, then collect that row's
            # cells. append sends the complete row to the output before the loop moves to the
            # next record.
            for field, value in zip(frame.columns, values):
                value = excel_value(value)
                cell = WriteOnlyCell(sheet, value=value)
                style_body_cell(cell, field, value, len(frame) > 10_000)
                row_cells.append(cell)
            sheet.append(row_cells)
            # Count each appended data row so the part length can be checked against the
            # plan. Heading rows are excluded from this evidence count.
            written_count += 1
        # If the iterator wrote fewer or more rows than planned, stop before claiming the
        # part is complete. The progress message reports data rows only.
        assert written_count == part["data_rows"], "An export part lost rows."
        print(f"Written {part['sheet_name']}: {written_count:,} data rows")
    # Save once after every planned sheet has been written. Both export functions produce one
    # workbook with the same full evidence and references.
    workbook.save(path)


### 12.5 Write the single workbook and check its sheets

**Question:** Was every planned table/partition saved with the correct headings?
Close an existing copy in Excel first. The temporary output becomes the final workbook only
after a successful write and header check. An error leaves the previous final workbook intact.
The validation above checks complete in-memory populations; this final read checks sheet names
and headings without rereading millions of evidence rows.


**Documentation:** [openpyxl.load_workbook](https://openpyxl.readthedocs.io/en/stable/api/openpyxl.reader.excel.html#openpyxl.reader.excel.load_workbook) ·
[Path.replace](https://docs.python.org/3.14/library/pathlib.html#pathlib.Path.replace).


In [ ]:
# Only replace the final workbook after writing and checking the new file successfully.

# Write to a .writing.xlsx name first. If export fails, an existing final workbook remains
# available rather than being replaced by an incomplete result.
temporary_path = OUTPUT_PATH.with_name(OUTPUT_PATH.stem + ".writing.xlsx")
# Run the writer selected from the export size or explicit setting. Both consume the same
# plan, so this choice changes memory use rather than evidence scope.
if selected_method == "pandas":
    export_with_pandas(temporary_path)
else:
    export_with_streaming(temporary_path)

# Reopen the candidate in read-only mode to inspect its sheet structure without loading
# millions of cells into ordinary worksheet objects. This step checks names and headings, not
# every saved evidence value.
saved = openpyxl.load_workbook(temporary_path, read_only=True, data_only=False)
try:
    # The actual sheet names and order must equal the complete plan. For each sheet, read
    # just its first row and compare it with the registered table's headings to detect
    # missing or mislabelled parts.
    assert saved.sheetnames == [part["sheet_name"] for part in export_plan]
    for part in export_plan:
        headings = next(saved[part["sheet_name"]].iter_rows(min_row=1, max_row=1, values_only=True))
        expected = tuple(tables[part["table_name"]].columns)
        assert headings == expected, f"{part['sheet_name']}: headings do not match the table."
# Close the read-only workbook even if an assertion fails. This also releases the file before
# renaming it on Windows.
finally:
    saved.close()

# This line is reached only if writing and the structure checks succeeded. It replaces a
# previous generated workbook, so preserve reusable edits in the notebook rather than relying
# on manual changes to the old Excel file.
temporary_path.replace(OUTPUT_PATH)
print("Saved:", OUTPUT_PATH.resolve())
print("Sheets:", len(export_plan), "| Issues:", len(issue_log), "| Checks:", len(check_log))


## 13. Use the workbook as an inspection record

Open `Workbook_Guide`, `Issue_Log` and `Check_Log` first. Follow `evidence_sheets` to the actual
evidence and use `Sheet_Index` if a large table has several parts. For example, an M02 issue
describes missing containment dates once; its evidence can contain hundreds of thousands of records.

Before moving to cleaning, check that your chosen analysis question, impact, severity reasoning,
proposed action and verification are justified by your current results. Keep valid extreme fires
and legitimate characters unless the source investigation supports another decision. Resolve
incident identity before treating repeated values as repeated physical fires.

**Study check:** Explain why an issue log can be short while its evidence is large. Explain why
R02 can retain more evidence rows than its affected count. Then explain why zero date-order failures
do not mean that records missing either date passed the date-order check.

**Troubleshooting:** If a package is missing, install it in the selected kernel using the guide at
the top. If the file is not found, check that `wildfires.csv` is beside this notebook. If Excel has
the output open, close it before exporting again. If a repeated check reference is reported,
restart the kernel and run from the top. For the large dataset, keep `EXPORT_METHOD = "auto"`.
